# DS5007 - Assignment 2
BBC Urdu NLP Pipeline

Student ID: 25i8030  
GitHub: https://github.com/Laibarao12/i25-8030-NLP-Assignment2

This public copy contains code only. The executed notebook and full BBC corpus are in the private course submission.

In [ ]:
# Assignment 2 · Cell 1 — Google Colab setup (run once)
import os, sys, subprocess, importlib.util
from pathlib import Path
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
ROOT = Path('/content/DS5007_Assignment2' if IN_COLAB else './DS5007_Assignment2').resolve()
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)
for folder in ['src','data','embeddings','models','figures','results','assets','tests']:
    (ROOT/folder).mkdir(exist_ok=True)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
needed = {'numpy':'numpy>=1.24,<3','scipy':'scipy>=1.10',
          'pandas':'pandas>=2.0','matplotlib':'matplotlib>=3.7',
          'sklearn':'scikit-learn>=1.3','torch':'torch>=2.0',
          'reportlab':'reportlab>=4.0','tqdm':'tqdm>=4.64'}
missing = [spec for mod,spec in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    print('Installing missing packages:', missing)
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
import torch
print('PROJECT ROOT:', ROOT)
print('PyTorch:', torch.__version__, '| GPU:', torch.cuda.is_available())
print('Standalone Assignment 2 environment ready.')

## Dataset

Upload `Assignment2_Dataset_300.zip` from Phase 0 when prompted. If Part 1 arrays already exist in this Colab runtime, the notebook reuses them. Otherwise Word2Vec training will run again.


In [ ]:
# Upload original Phase 0 outputs into this separate Assignment 2 notebook
from pathlib import Path
import io, zipfile

DATA_DIR = ROOT / 'data'
DATA_DIR.mkdir(exist_ok=True)
REQUIRED = ('raw.txt', 'cleaned.txt', 'Metadata.json')
OPTIONAL = ('topic_labels_review.csv', 'corpus_audit.json')
ALLOWED = set(REQUIRED + OPTIONAL)

if all((DATA_DIR / item).is_file() for item in REQUIRED):
    print('Dataset files already available. Upload skipped.')
else:
    try:
        from google.colab import files
    except ImportError:
        raise RuntimeError('Upload Phase 0 dataset files into the data/ folder before continuing.')
    print('Choose Assignment2_Dataset_300.zip or Phase0_BBC_Urdu_Data_Backup.zip')
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError('No dataset uploaded. Required: raw.txt, cleaned.txt, Metadata.json')
    imported = set()
    for filename, content in uploaded.items():
        if filename.lower().endswith('.zip'):
            with zipfile.ZipFile(io.BytesIO(content)) as archive:
                for info in archive.infolist():
                    if info.is_dir():
                        continue
                    base = Path(info.filename).name
                    if base not in ALLOWED:
                        continue
                    if base in imported:
                        raise ValueError('Dataset ZIP contains multiple entries named '+base)
                    (DATA_DIR / base).write_bytes(archive.read(info))
                    imported.add(base)
        else:
            base = Path(filename).name
            if base not in ALLOWED:
                print('Ignored unrelated file:', filename)
                continue
            (DATA_DIR / base).write_bytes(content)
            imported.add(base)
    print('Imported:', sorted(imported))

missing = [item for item in REQUIRED if not (DATA_DIR / item).is_file()]
if missing:
    raise FileNotFoundError('Dataset still missing: '+', '.join(missing))
print('READY: all three dataset files are present.')
for name in REQUIRED + OPTIONAL:
    path = DATA_DIR / name
    if path.exists(): print(f'{name}: {path.stat().st_size:,} bytes')

## Setup
These cells create the Python modules used in the experiments. Run them once.

In [ ]:
%%writefile src/__init__.py
from . import common, embeddings

In [ ]:
%%writefile src/a1_linguistics.py
"""Original Urdu linguistic rules reproduced directly from the supplied Assignment 1 notebook.
Functions are A1-authored, not pretrained. Preserve original behavior for reproducibility.
Known shortcomings: some suffix stemming is too aggressive and percentages may remain literal.
"""
import re

ARABIC_DIACRITICS = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED\u08F0-\u08FF]")

def remove_diacritics(text):
    return ARABIC_DIACRITICS.sub("", text)

URL_RE = re.compile(r"(https?://\S+|www\.\S+)")

EMOJI_RE = re.compile(
    "["
    "\U0001F300-\U0001FAFF"
    "\U00002600-\U000027BF"
    "\U0001F1E0-\U0001F1FF"
    "]+", flags=re.UNICODE
)

ZERO_WIDTH_RE = re.compile(r"[\u200B\u200C\u200D\uFEFF]")

LATIN_RE = re.compile(r"[A-Za-z]+")

URDU_KEEP_RE = re.compile(
    r"[^\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF\uFB50-\uFDFF\uFE70-\uFEFF"
    r"\u06F0-\u06F90-9\s۔،؟!:٪\-]"
)

def remove_noise(text):
    text = URL_RE.sub(" ", text)
    text = EMOJI_RE.sub(" ", text)
    text = ZERO_WIDTH_RE.sub(" ", text)
    text = LATIN_RE.sub(" ", text)
    return text

def remove_non_urdu(text):
    return URDU_KEEP_RE.sub(" ", text)

SENTENCE_SPLIT_RE = re.compile(r"([^۔؟!]*[۔؟!])")

def segment_sentences(text):
    sentences = [s.strip() for s in SENTENCE_SPLIT_RE.findall(text) if s.strip()]
    remainder = SENTENCE_SPLIT_RE.sub("", text).strip()
    if remainder:
        sentences.append(remainder)
    return sentences

WHITESPACE_RE = re.compile(r"\s+")

def normalize_whitespace(text):
    return WHITESPACE_RE.sub(" ", text).strip()

PUNCTUATION = set("۔،؟!:٪-()[]{}\"'")

POSTPOSITIONS = {"کا", "کی", "کے", "میں", "پر", "سے", "تک", "کو", "نے", "ہی", "بھی", "تو", "لیے", "والا", "والی", "والے"}

NUM_RE = re.compile(r"^[0-9\u06F0-\u06F9]+$")

def urdu_tokenize(text):
    tokens = []
    for chunk in text.split():
        if not chunk:
            continue
        # peel off leading punctuation
        while chunk and chunk[0] in PUNCTUATION:
            tokens.append(chunk[0])
            chunk = chunk[1:]
        # peel off trailing punctuation (may be more than one, e.g. word۔)
        trailing = []
        while chunk and chunk[-1] in PUNCTUATION:
            trailing.append(chunk[-1])
            chunk = chunk[:-1]
        if chunk:
            if NUM_RE.match(chunk):
                tokens.append("<NUM>")
            else:
                tokens.append(chunk)
        tokens.extend(reversed(trailing))
    return tokens

URDU_SUFFIXES = sorted(
    ["وں", "یں", "ات", "ائیں", "ائی", "گی", "گا", "گے", "تا", "تی", "تے", "ہا", "یاں", "ے", "ی", "ہ"],
    key=len, reverse=True,
)

def urdu_stem(word):
    if word in {"<NUM>"} or word in PUNCTUATION or word in POSTPOSITIONS:
        return word
    for suf in URDU_SUFFIXES:
        if word.endswith(suf) and len(word) - len(suf) >= 2:
            return word[: -len(suf)]
    return word

LEMMA_DICT = {
    "لڑکے": "لڑکا", "لڑکوں": "لڑکا",
    "لوگوں": "لوگ", "بچوں": "بچہ", "بچے": "بچہ",
    "کتابیں": "کتاب", "کتابوں": "کتاب",
    "شہروں": "شہر", "علاقوں": "علاقہ",
    "کھلاڑیوں": "کھلاڑی", "کسانوں": "کسان",
    "مریضوں": "مریض", "شرکاء": "شریک",
}

def urdu_lemmatize(word):
    if word in {"<NUM>"} or word in PUNCTUATION or word in POSTPOSITIONS:
        return word
    if word in LEMMA_DICT:
        return LEMMA_DICT[word]
    # regular oblique-plural pattern: X + وں  ->  X + ا   (e.g. گھروں -> گھرا is imperfect; fall back safely)
    if word.endswith("وں") and len(word) > 3:
        return word[:-2]
    if word.endswith("یاں") and len(word) > 3:
        return word[:-3] + "ی"
    return word

def clean_and_process_article(raw_text):
    text = remove_diacritics(raw_text)
    text = remove_noise(text)
    text = remove_non_urdu(text)
    text = normalize_whitespace(text)
    sentences = segment_sentences(text)

    processed_sentences = []
    for sent in sentences:
        tokens = urdu_tokenize(sent)
        tokens = [urdu_stem(t) for t in tokens]
        tokens = [urdu_lemmatize(t) for t in tokens]
        if tokens:
            processed_sentences.append(" ".join(tokens))
    return processed_sentences

In [ ]:
%%writefile src/common.py
"""Corpus parsing, repeatable splits, plots, and results logging for DS5007 A2.
No pretrained weights, Gensim, HuggingFace or built-in Transformer modules.
"""
from __future__ import annotations
import json, re, random, unicodedata, os
from pathlib import Path
from collections import Counter
import numpy as np
import torch
from sklearn.model_selection import train_test_split
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

ROOT = Path(__file__).resolve().parents[1]
DATA = ROOT / 'data'
EMB = ROOT / 'embeddings'
MODELS = ROOT / 'models'
FIGS = ROOT / 'figures'
RES = ROOT / 'results'
for p in [DATA, EMB, MODELS, FIGS, RES]: p.mkdir(parents=True, exist_ok=True)
TOPICS = ['Politics', 'Sports', 'Economy', 'International', 'Health & Society']
STOP = {'<pad>':0,'<unk>':1}
URDU_DIACRITICS = re.compile('[\u064b-\u065f\u0670\u06d6-\u06ed]')
TOK_RE = re.compile(r"<num>|[\u0621-\u06d3\u06fa-\u06ff]+|[A-Za-z]+(?:['’][A-Za-z]+)?|\d+(?:[.,]\d+)*|[^\w\s]", re.UNICODE)
SENT_RE = re.compile(r'(?<=[.!?؟۔])\s+|\n+')


def seed_all(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)


def normalize(s):
    s = unicodedata.normalize('NFKC', str(s)).lower()
    s = URDU_DIACRITICS.sub('', s)
    return s.replace('ي','ی').replace('ك','ک').replace('\u0640','').strip()


def tokenize(s, keep_punc=False):
    ts = TOK_RE.findall(normalize(s))
    return ts if keep_punc else [t for t in ts if any(ch.isalnum() for ch in t)]


def load_text(name):
    path = DATA/name
    if not path.exists():
        raise FileNotFoundError(f'Missing required file: {path}. Copy your actual Assignment 1 corpus here.')
    return path.read_text('utf-8-sig', errors='replace')


def load_metadata():
    path=DATA/'Metadata.json'
    if not path.exists(): raise FileNotFoundError(f'Missing required file: {path}')
    x=json.loads(path.read_text('utf-8-sig'))
    if isinstance(x,list): return x
    if isinstance(x,dict):
        for key in ['articles','documents','records','data','items','news']:
            if key in x and isinstance(x[key],list): return x[key]
        if all(isinstance(v,dict) for v in x.values()): return list(x.values())
    raise ValueError('Unexpected Metadata.json structure; revise load_metadata() to match the real source.')


def label_topic(rec):
    """Choose five-class labels only from unambiguous metadata or explicit title keywords.

    BBC's scraped category field frequently contains a *site-wide navigation breadcrumb*.
    Those breadcrumbs are not category annotations and must not be trusted.
    """
    if isinstance(rec,str): rec={'category':rec}
    patterns={
      'Politics':['politic','government','election','parliament','siyasat','hukumat','assembly','سیاس','حکومت','انتخاب'],
      'Sports':['sport','cricket','football','khel','match','کھیل','کرکٹ'],
      'Economy':['econom','business','finance','maeeshat','budget','trade','معیش','کاروبار','اقتصاد'],
      'International':['international','world','foreign','dunya','عالمی','بین الاقوامی','خارجہ','دنیا'],
      'Health & Society':['health','society','education','sehat','taleem','social','science','صحت','تعلیم','معاشر'],
    }
    # Prioritize clearly labeled topic/section fields; don't assume page navigation is a label.
    for key in ('topic','label','class','news_category','genre','section','category'):
        value=str(rec.get(key,'')).strip()
        if not value:continue
        low=value.lower()
        # Original Assignment 1's BBC breadcrumb is typically an entire navigation bar.
        if ('صفحۂ اول' in value or 'ویڈیو سیکشن' in value or
            len(value.split())>9 or ',' in value):continue
        for label in TOPICS:
            if low==label.lower():return label,'metadata'
        hits=[topic for topic,patterns_for_topic in patterns.items()
              if any(word in low for word in patterns_for_topic)]
        if len(hits)==1:return hits[0],'metadata'
    # Title is only weak supervision: a human must validate the review file before grading.
    txt=' '.join(str(rec.get(k,'')) for k in ('title','headline')).lower()
    kw={
       'Politics':['election','government','minister','parliament','imran','hukumat','انتخاب','حکومت','وزیر','پارلیمان'],
       'Sports':['cricket','match','team','player','score','khel','کرکٹ','میچ','کھلاڑی','ٹیم'],
       'Economy':['inflation','trade','bank','gdp','budget','maeeshat','مہنگائی','تجارت','بینک','بجٹ'],
       'International':['united nations','treaty','foreign','bilateral','conflict','اقوام متحدہ','معاہدہ','خارجہ','سرحد'],
       'Health & Society':['hospital','disease','vaccine','flood','education','ہسپتال','بیماری','ویکسین','سیلاب','تعلیم'],
    }
    counts={k:sum(txt.count(w) for w in words) for k,words in kw.items()}
    if max(counts.values())>0:
        top=max(counts.values());winners=[k for k,n in counts.items() if n==top]
        if len(winners)==1:return winners[0],'title_keyword_weak_label'
    return None,'unlabelled'


def split_numbered_articles(text):
    """Return [(article_id, body)] from the exact A1 '### ARTICLE N ###' structure.

    Return None for unrelated formats. Don't merge/split on blank lines inside articles.
    """
    pattern=r'(?m)^### ARTICLE\s+(\d+)\s+###\s*$'
    m=list(re.finditer(pattern,text))
    if not m:return None
    if text[:m[0].start()].strip():
        raise ValueError('Unexpected content before first A1 article marker.')
    out=[];seen=set()
    for i,match in enumerate(m):
        number=int(match.group(1))
        if number in seen:raise ValueError(f'Duplicate A1 article marker: {number}')
        seen.add(number)
        body=text[match.end():m[i+1].start() if i+1<len(m) else len(text)].strip()
        if not body:raise ValueError(f'A1 article {number} is empty')
        out.append((number,body))
    return out


def load_articles(require_labels=True):
    """Align cleaned A1 article markers with Metadata.json IDs (not arbitrary row order).

    Optional data/topic_labels_review.csv manual topic overrides take precedence. This
    file must contain columns article_number and topic; no fake topics are created.
    """
    import csv
    text=load_text('cleaned.txt'); records=load_metadata()
    numbered=split_numbered_articles(text)
    by_id={}
    if numbered is not None:
        if len({k for k,_ in numbered})!=len(numbered):raise ValueError('Duplicate article ID')
        by_id=dict(numbered)
        if not all(isinstance(r,dict) and 'article_number' in r for r in records):
            raise ValueError('A1 numbered corpus needs article_number in every Metadata.json record')
        metadata_ids=[int(r['article_number']) for r in records]
        if len(set(metadata_ids))!=len(metadata_ids):raise ValueError('Duplicate metadata article_number')
        if set(metadata_ids)!=set(by_id):
            raise ValueError(f'A1 article ID mismatch. Only in cleaned: {sorted(set(by_id)-set(metadata_ids))[:10]}; '
                             f'only in metadata: {sorted(set(metadata_ids)-set(by_id))[:10]}')
        sep='A1 numbered article blocks with verified IDs'
    else:
        by_line=[s.strip() for s in text.splitlines() if s.strip()]
        by_blank=[s.strip().replace('\n',' ') for s in re.split(r'\n\s*\n',text) if s.strip()]
        if len(by_line)==len(records):docs=by_line;sep='one line per article'
        elif len(by_blank)==len(records):docs=by_blank;sep='blank-line-separated articles'
        else:
            raise ValueError(f'Cannot align corpus with metadata: {len(by_line)} non-empty lines, '
                             f'{len(by_blank)} blank blocks, {len(records)} metadata records')
        records=[dict(r,article_number=r.get('article_number',i+1)) if isinstance(r,dict) else
                 {'article_number':i+1,'category':str(r)} for i,r in enumerate(records)]
        by_id={int(r['article_number']):doc for r,doc in zip(records,docs)}
    overrides={}
    path=DATA/'topic_labels_review.csv'
    if path.exists():
        with path.open(encoding='utf-8-sig',newline='') as f:
            for row in csv.DictReader(f):
                aid=row.get('article_number','').strip()
                if not aid:continue
                topic=row.get('topic','').strip()
                if topic:
                    if topic not in TOPICS:raise ValueError(f'Invalid topic override {topic} for article {aid}')
                    overrides[int(aid)]=topic
        extra=set(overrides)-set(by_id)
        if extra:raise ValueError(f'Unknown article numbers in topic labels: {sorted(extra)}')
    out=[];sources=[]
    for r in records:
        article_id=int(r['article_number'])
        if article_id in overrides:label,source=overrides[article_id],'manual_review'
        else:label,source=label_topic(r)
        sources.append(source)
        if require_labels and label is None:continue
        s=by_id[article_id]
        out.append({'id':article_id,'text':s,'tokens':tokenize(s),'topic':label,'label_source':source})
    if require_labels and not out:raise ValueError('No labelled articles; review data/topic_labels_review.csv')
    print(f'Loaded {len(out)}/{len(records)} aligned articles ({sep}); '
          f'label sources: {dict(Counter(sources))}')
    return out


def vocab_from_docs(docs,max_tokens=10000):
    counts=Counter(t for doc in docs for t in doc)
    vocab=dict(STOP)
    for t,_ in counts.most_common(max_tokens):
        if t not in vocab:vocab[t]=len(vocab)
    return vocab,counts


def stratified_split(items,labels,seed=42):
    inds=np.arange(len(items)); counts=Counter(labels)
    if len(counts)<2 or min(counts.values())<7:
        raise ValueError(f'Not enough examples per category for stratified 70/15/15: {counts}')
    tr,temp=train_test_split(inds,test_size=.30,stratify=labels,random_state=seed)
    va,te=train_test_split(temp,test_size=.5,stratify=np.asarray(labels)[temp],random_state=seed)
    return [list(map(int,x)) for x in (tr,va,te)]


def save_json(name,obj):
    path=RES/name
    path.write_text(json.dumps(obj,ensure_ascii=False,indent=2,default=str),'utf-8')
    return path


def show_counts(name, labels, indices=None):
    counts=Counter(labels[i] for i in indices) if indices is not None else Counter(labels)
    print(f'{name}:', dict(sorted(counts.items())))
    return dict(counts)


def plot_curves(rows, keys, name, title):
    fig,ax=plt.subplots(figsize=(8,4.5))
    for key in keys: ax.plot([r['epoch'] for r in rows], [r.get(key,np.nan) for r in rows],marker='o',markersize=3,label=key)
    ax.set(xlabel='Epoch',ylabel='Metric value',title=title)
    ax.legend();ax.grid(alpha=.2); fig.tight_layout();fig.savefig(FIGS/name,dpi=170);plt.close(fig)


def plot_confusion(cm,labels,name,title):
    fig,ax=plt.subplots(figsize=(8,6))
    im=ax.imshow(cm,cmap='Blues');fig.colorbar(im,ax=ax)
    ax.set_xticks(range(len(labels)), labels, rotation=45,ha='right')
    ax.set_yticks(range(len(labels)), labels)
    ax.set(xlabel='Predicted label',ylabel='True label',title=title)
    if len(labels)<=6:
        for i in range(len(labels)):
            for j in range(len(labels)):ax.text(j,i,int(cm[i,j]),ha='center',va='center',fontsize=8)
    fig.tight_layout();fig.savefig(FIGS/name,dpi=170);plt.close(fig)


def device():return torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
%%writefile src/embeddings.py
"""Part 1: exact TF-IDF, symmetric PPMI, PyTorch Skip-gram/negative sampling."""
from __future__ import annotations
from collections import Counter
import json, itertools, math, time
from pathlib import Path
import numpy as np
from scipy import sparse
from scipy.sparse.linalg import norm as sparse_norm
import torch
from torch import nn
import torch.nn.functional as F
from sklearn.manifold import TSNE
from sklearn.decomposition import TruncatedSVD
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from .common import (DATA, EMB, FIGS, RES, TOPICS, tokenize, load_text, load_articles,
                     vocab_from_docs, seed_all, save_json, device, split_numbered_articles, normalize)

WINDOW=5; K=10; BATCH=1024; EPOCHS=5
QUERIES=['Pakistan','Hukumat','Adalat','Maeeshat','Fauj','Sehat','Taleem','Aabadi']
SEED_SEMANTICS={
    'Politics':'pakistan hukumat adalat wazeer wazir intikhab assembly election government parliament سیاست حکومت عدالت وزیر انتخاب پاکستان'.split(),
    'Sports':'cricket match team player football captain score wicket کرکٹ میچ ٹیم کھلاڑی وکٹ کھیل'.split(),
    'Economy':'maeeshat bank business tijarat economy trade inflation budget budget معیشت تجارت بینک بجٹ مہنگائی'.split(),
    'International':'afghanistan india china iran america world united nations foreign افغانستان بھارت چین ایران امریکہ عالمی'.split(),
    'Health & Society':'sehat taleem school hospital doctor vaccine health education hospital صحت تعلیم اسکول ہسپتال ڈاکٹر'.split(),
}


def docs_from_text(name):
    text=load_text(name)
    numbered=split_numbered_articles(text)
    chunks=[body for _,body in numbered] if numbered is not None else [s.strip() for s in text.splitlines() if s.strip()]
    return [tok for chunk in chunks if (tok:=tokenize(chunk))]


def save_dense_npy_from_csr(mat,path):
    arr=np.lib.format.open_memmap(path,mode='w+',dtype='float32',shape=mat.shape)
    arr[:]=0
    for start in range(0,mat.shape[0],256):
        end=min(start+256,mat.shape[0]);arr[start:end]=mat[start:end].toarray().astype('float32')
    arr.flush();del arr


def tfidf(articles,vocab):
    n=len(articles); rows=[];cols=[];vals=[]
    for i,article in enumerate(articles):
        c=Counter(vocab.get(t,1) for t in article['tokens'])
        for w,count in c.items():rows.append(i);cols.append(w);vals.append(count)
    x=sparse.csr_matrix((np.asarray(vals,dtype=np.float32),(rows,cols)),shape=(n,len(vocab)))
    df=np.asarray((x>0).sum(axis=0)).ravel()
    idf=np.log(n/(1.+df))  # exactly the assignment formula, not sklearn's smoothed version
    weighted=x.multiply(idf).tocsr()
    save_dense_npy_from_csr(weighted,EMB/'tfidf_matrix.npy')
    y=[a['topic'] for a in articles]
    inverse={v:k for k,v in vocab.items()}
    result={}
    for topic in TOPICS:
        sel=np.where(np.asarray(y)==topic)[0]
        if not len(sel):result[topic]=[];continue
        mean=np.asarray(weighted[sel].mean(axis=0)).ravel()
        rest=np.where(np.asarray(y)!=topic)[0]
        others=np.asarray(weighted[rest].mean(axis=0)).ravel() if len(rest) else np.zeros_like(mean)
        score=mean-others;score[:2]=-np.inf
        top=np.argsort(score)[::-1][:10]
        result[topic]=[{'word':inverse[int(i)],'topic_mean_tfidf':round(float(mean[i]),5),
                        'topic_minus_others':round(float(score[i]),5)} for i in top]
        print(topic, 'top 10:', [inverse[int(i)] for i in top])
    save_json('tfidf_top_words.json',result)
    print('Saved TF-IDF', weighted.shape, 'nonzeros:',weighted.nnz)
    return result


def ppmi_matrix(docs,vocab):
    """Directed symmetric +/- k counts in sparse CSR, saved as required dense .npy.
    Streaming COO->CSR chunks avoid holding all corpus pair arrays simultaneously.
    """
    v=len(vocab); acc=sparse.csr_matrix((v,v),dtype=np.float64)
    ii=[];jj=[]; chunk_target=1_500_000
    def flush():
        nonlocal acc,ii,jj
        if not ii:return
        batch=sparse.coo_matrix((np.ones(len(ii),dtype=np.float64),(ii,jj)),shape=(v,v)).tocsr()
        acc=acc+batch;ii.clear();jj.clear()
    for d in docs:
        ids=[vocab.get(t,1) for t in d]
        for ix,center in enumerate(ids):
            for j in range(max(0,ix-WINDOW),min(len(ids),ix+WINDOW+1)):
                if j!=ix:ii.append(center);jj.append(ids[j])
            if len(ii)>chunk_target:flush()
    flush()
    acc.eliminate_zeros()
    total=acc.sum()
    row=np.asarray(acc.sum(axis=1)).ravel(); col=np.asarray(acc.sum(axis=0)).ravel()
    coo=acc.tocoo()
    # log2(C_ij * total / (row_i * col_j)) for nonzero counts.
    ratio=np.asarray(coo.data*total/(row[coo.row]*col[coo.col]+1.e-30),dtype=np.float64)
    values=np.maximum(np.log2(np.maximum(ratio,1e-30)),0.)
    ppmi=sparse.csr_matrix((values.astype(np.float32),(coo.row,coo.col)),shape=(v,v))
    ppmi.eliminate_zeros()
    save_dense_npy_from_csr(ppmi,EMB/'ppmi_matrix.npy')
    print('Saved PPMI',ppmi.shape,'nnz',ppmi.nnz)
    return ppmi


# Assignment questions name queries in Roman Urdu but the A1 preprocessing
# deliberately removes Latin script. Translate for lookup, not for training.
ROMAN_QUERY_ALIASES={
    'pakistan':'پاکستان','hukumat':'حکومت','adalat':'عدالت',
    'maeeshat':'معیشت','fauj':'فوج','sehat':'صحت',
    'taleem':'تعلیم','aabadi':'آبادی',
    'islamabad':'اسلام آباد','karachi':'کراچی',
}


def resolve_word(word,vocab):
    """Find the corpus *surface or A1-stemmed* representation of one phrase.

    No word is inserted into the vocabulary for evaluation. Returns None
    if the word was not actually observed.
    """
    from .a1_linguistics import clean_and_process_article, urdu_stem, urdu_lemmatize
    raw=str(word).strip()
    translated=ROMAN_QUERY_ALIASES.get(raw.lower(),raw)
    candidates=[normalize(raw), normalize(translated)]
    for cand in (translated,):
        try:
            processed=clean_and_process_article(cand)
            if len(processed)==1 and len(processed[0].split())==1:
                candidates.append(normalize(processed[0]))
            candidates += [normalize(urdu_stem(cand)), normalize(urdu_lemmatize(cand))]
        except Exception:
            pass
    # Multi-word place names may be separate tokens; corpus is word-level.
    # Prefer a whole phrase only if the tokenizer did produce that exact token.
    for c in candidates:
        if c in vocab:return c
    return None



def resolve_indices(word,vocab):
    """Return in-vocab token ids, averaging word vectors for multiword names."""
    simple=resolve_word(word,vocab)
    if simple is not None:return [vocab[simple]]
    from .a1_linguistics import clean_and_process_article
    translated=ROMAN_QUERY_ALIASES.get(str(word).lower().strip(),str(word))
    clean=' '.join(clean_and_process_article(translated))
    parts=tokenize(clean)
    if len(parts)>=2 and all(t in vocab for t in parts):
        return [vocab[t] for t in parts]
    return None


def phrase_vector(emb, vocab, word):
    ids=resolve_indices(word,vocab)
    if ids is None:return None,None
    if sparse.issparse(emb):
        return np.asarray(emb[ids].mean(axis=0)).ravel(),ids
    return np.asarray(emb[ids],dtype=np.float32).mean(axis=0),ids

def cosine_neighbors(emb,vocab,word,n=10):
    vec,ids=phrase_vector(emb,vocab,word)
    if vec is None:return []
    norms=np.sqrt(np.asarray(emb.power(2).sum(axis=1)).ravel())+1e-12 if sparse.issparse(emb) else np.linalg.norm(emb,axis=1)+1e-9
    scores=np.asarray(emb@vec).ravel()/(norms*(np.linalg.norm(vec)+1e-9))
    for idx in ids:scores[idx]=-np.inf
    scores[:2]=-np.inf
    inv={v:k for k,v in vocab.items()}
    top=np.argsort(scores)[::-1][:n]
    return [(inv.get(int(i),'?'),round(float(scores[i]),4)) for i in top if np.isfinite(scores[i])]


def plot_tsne(ppmi,vocab,counts):
    inv={i:t for t,i in vocab.items()}
    top=[vocab[t] for t,_ in counts.most_common(200) if t in vocab and vocab[t]>1][:200]
    if len(top)<10:print('Too few tokens for t-SNE');return
    subset=ppmi[top].toarray()
    # SVD first provides an efficient, noise-reduced embedding for t-SNE.
    svd=TruncatedSVD(n_components=min(50,subset.shape[0]-1,subset.shape[1]-1),random_state=42)
    dense=svd.fit_transform(subset)
    points=TSNE(n_components=2,perplexity=min(30,(len(top)-1)//3),random_state=42,init='pca',learning_rate='auto').fit_transform(dense)
    name_map={w.lower():c for c,terms in SEED_SEMANTICS.items() for w in terms}
    labels=[name_map.get(inv[i].lower(),'Other / unlabelled') for i in top]
    palette=['#b54c5e','#256e96','#d28e3b','#6a52a4','#36917a','#878787']
    fig,ax=plt.subplots(figsize=(13,9))
    for cl,color in zip(TOPICS+['Other / unlabelled'],palette):
        mask=np.array([x==cl for x in labels]);
        if mask.any():ax.scatter(points[mask,0],points[mask,1],s=34,alpha=.76,label=cl,c=color)
    for i,wid in enumerate(top):ax.annotate(inv[wid],points[i],fontsize=6,alpha=.8)
    ax.set(xlabel='t-SNE dimension 1',ylabel='t-SNE dimension 2',title='PPMI: 200 frequent tokens (manual semantic seed labels)')
    ax.legend(loc='best',fontsize=9);fig.tight_layout();fig.savefig(FIGS/'ppmi_tsne.png',dpi=200);plt.close(fig)
    print('Saved semantic t-SNE plot; unlabeled tokens are not assumed to belong to a topic')


class SkipGram(nn.Module):
    def __init__(self,n,d):
        super().__init__();self.V=nn.Embedding(n,d,padding_idx=0);self.U=nn.Embedding(n,d,padding_idx=0)
        nn.init.uniform_(self.V.weight,-.5/d,.5/d);nn.init.zeros_(self.U.weight)
        with torch.no_grad():self.V.weight[0].zero_()
    def forward(self,centers,contexts,negatives):
        v=self.V(centers)
        pos=(v*self.U(contexts)).sum(dim=1)
        neg=torch.bmm(self.U(negatives),v.unsqueeze(2)).squeeze(2)
        return -(F.logsigmoid(pos)+F.logsigmoid(-neg).sum(dim=1)).mean()
    def mean_embeddings(self):
        return ((self.V.weight.detach()+self.U.weight.detach())*.5).cpu().numpy().astype(np.float32)


def pairs(docs,vocab,seed):
    """On-the-fly positive pairs; shuffle docs each epoch, fixed symmetric k=5."""
    rng=np.random.default_rng(seed)
    order=rng.permutation(len(docs))
    for di in order:
        ids=[vocab.get(t,1) for t in docs[di]]
        for ix in rng.permutation(len(ids)):
            w=ids[ix]
            if w==0:continue
            for j in range(max(0,ix-WINDOW),min(len(ids),ix+WINDOW+1)):
                if j!=ix and ids[j]!=0:yield (w,ids[j])


def train_sg(docs,vocab,counts,d,filename,epochs=EPOCHS, max_batches=None):
    dev=device();seed_all(42)
    print(f'Train {filename} | articles={len(docs)} | vocabulary={len(vocab)} | dim={d} | device={dev}')
    freqs=np.zeros(len(vocab),dtype=np.float64)
    for w,n in counts.items():freqs[vocab.get(w,1)]+=n
    freqs[:1]=0
    prob=np.power(freqs,.75);prob/=prob.sum()
    assert np.isclose(prob.sum(),1)
    prob_t=torch.tensor(prob,dtype=torch.float32,device=dev)
    model=SkipGram(len(vocab),d).to(dev);opt=torch.optim.Adam(model.parameters(),lr=.001)
    history=[];step=0
    for epoch in range(1,epochs+1):
        start=time.time();total_loss=0.;num_batches=0;buf=[]
        def update(buf):
            nonlocal step,total_loss,num_batches
            center=torch.tensor([a for a,b in buf],device=dev,dtype=torch.long)
            context=torch.tensor([b for a,b in buf],device=dev,dtype=torch.long)
            neg=torch.multinomial(prob_t,num_samples=len(buf)*K,replacement=True).reshape(-1,K)
            opt.zero_grad(set_to_none=True)
            loss=model(center,context,neg);loss.backward();opt.step()
            total_loss+=float(loss.item());num_batches+=1;step+=1
            if step%100==0:print(f'  epoch={epoch}, step={step}, batch loss={loss.item():.4f}')
        for pair in pairs(docs,vocab,seed=42+epoch):
            buf.append(pair)
            if len(buf)>=BATCH:
                update(buf);buf=[]
                if max_batches is not None and num_batches>=max_batches:break
        if buf and (max_batches is None or num_batches<max_batches):update(buf)
        if not num_batches:raise ValueError('No skip-gram training pairs available')
        mean=total_loss/num_batches
        history.append({'epoch':epoch,'train_loss':mean,'batches':num_batches,'seconds':round(time.time()-start,2)})
        print(f'epoch {epoch}/{epochs} average loss {mean:.4f} over {num_batches} batches')
    np.save(EMB/filename,model.mean_embeddings())
    fig,ax=plt.subplots(figsize=(7,4));ax.plot([x['epoch'] for x in history],[x['train_loss'] for x in history],marker='o')
    ax.set(xlabel='Epoch',ylabel='Mean skip-gram negative sampling loss',title=f'Skip-gram training loss: {filename}')
    ax.grid(alpha=.2);fig.tight_layout();fig.savefig(FIGS/f'{Path(filename).stem}_loss.png',dpi=170);plt.close(fig)
    save_json(f'{Path(filename).stem}_training.json',history)
    return model.mean_embeddings(),history


def analogy_results(emb,vocab):
    tests=json.loads((Path(__file__).resolve().parents[1]/'assets'/'analogies.json').read_text('utf-8'))
    arr=emb.toarray() if sparse.issparse(emb) else np.asarray(emb)
    norms=np.linalg.norm(arr,axis=1)+1e-9
    inv={v:k for k,v in vocab.items()};rows=[]
    for r in tests:
        a,ai=phrase_vector(arr,vocab,r['a']);b,bi=phrase_vector(arr,vocab,r['b']);c,ci=phrase_vector(arr,vocab,r['c'])
        gold=resolve_word(r['gold'],vocab)
        if any(x is None for x in (a,b,c)):
            rows.append({**r,'candidates':[],'correct_top3':None,'status':'OOV premise'});continue
        target=b-a+c
        score=arr@target/(norms*(np.linalg.norm(target)+1e-9))
        for idx in set(ai+bi+ci):score[idx]=-np.inf
        score[:2]=-np.inf
        rank=np.argsort(score)[::-1][:3]
        candidates=[inv[int(i)] for i in rank]
        rows.append({**r,'candidates':candidates,'correct_top3':(gold in candidates) if gold is not None else None,
                     'status':'evaluated' if gold is not None else 'gold OOV'})
    return rows


def mrr_20(emb,vocab):
    test=json.loads((Path(__file__).resolve().parents[1]/'assets'/'mrr_20_pairs.json').read_text('utf-8'))
    if len(test)!=20:raise ValueError('MRR gold file must have 20 human-labelled word pairs')
    arr=emb.toarray() if sparse.issparse(emb) else np.asarray(emb)
    norms=np.linalg.norm(arr,axis=1)+1e-9
    scored=[]
    for r in test:
        query,exclude=phrase_vector(arr,vocab,r['query']); gold=resolve_word(r['gold'],vocab)
        if query is None or gold is None:
            scored.append({**r,'reciprocal_rank':None,'status':'OOV'});continue
        score=arr@query/(norms*(np.linalg.norm(query)+1e-9))
        for idx in exclude:score[idx]=-np.inf
        score[:2]=-np.inf
        rank=int(np.sum(score>score[vocab[gold]])+1)
        scored.append({**r,'rank':rank,'reciprocal_rank':1./rank,'status':'evaluated'})
    valid=[x['reciprocal_rank'] for x in scored if x['reciprocal_rank'] is not None]
    return {'MRR_evaluable':float(np.mean(valid)) if valid else None,
            'evaluable_pairs':len(valid),'requested_pairs':len(test),'detail':scored}


def evaluate_conditions(mat,vocs):
    query_words=QUERIES+['islamabad','karachi']
    out={}
    for cond,emb in mat.items():
        vocab=vocs[cond]
        neigh={q:cosine_neighbors(emb,vocab,q,5) for q in query_words[:5]}
        metric=mrr_20(emb,vocab)
        out[cond]={'neighbors_five_queries':neigh,'MRR':metric['MRR_evaluable'],
                   'evaluable_pairs':metric['evaluable_pairs'],'total_pairs':20}
        print(cond,'MRR:',metric['MRR_evaluable'],'eligible:',metric['evaluable_pairs'])
        for q,v in neigh.items():print(' ',q,v)
    save_json('four_condition_comparison.json',out)
    return out


def run():
    seed_all(42)
    articles=load_articles(require_labels=False)
    cleaned=[a['tokens'] for a in articles]
    raw=docs_from_text('raw.txt')
    vocab,counts=vocab_from_docs(cleaned,10000)
    (EMB/'word2idx.json').write_text(json.dumps(vocab,ensure_ascii=False,indent=2),'utf-8')
    topwords=tfidf(articles,vocab)
    ppmi=ppmi_matrix(cleaned,vocab)
    plot_tsne(ppmi,vocab,counts)
    p_neighbors={q:cosine_neighbors(ppmi,vocab,q,5) for q in QUERIES+['islamabad','karachi']}
    save_json('ppmi_neighbors.json',p_neighbors)
    print('\nPPMI neighbors:');[print(k,v) for k,v in p_neighbors.items()]
    rawv,rawcounts=vocab_from_docs(raw,10000)
    (EMB/'raw_word2idx.json').write_text(json.dumps(rawv,ensure_ascii=False,indent=2),'utf-8')
    c2,_=train_sg(raw,rawv,rawcounts,100,'embeddings_w2v_raw.npy')
    c3,_=train_sg(cleaned,vocab,counts,100,'embeddings_w2v.npy')
    c4,_=train_sg(cleaned,vocab,counts,200,'embeddings_w2v_200.npy')
    queries={q:cosine_neighbors(c3,vocab,q,10) for q in QUERIES}
    save_json('w2v_neighbors.json',queries)
    print('Required eight query words:');[print(k,v) for k,v in queries.items()]
    analogies=analogy_results(c3,vocab);save_json('analogies.json',analogies)
    print('Analogies (top3 correct / evaluable):',sum(x['correct_top3'] is True for x in analogies),'/',sum(x['correct_top3'] is not None for x in analogies))
    comparison=evaluate_conditions({'C1_PPMI':ppmi,'C2_raw':c2,'C3_cleaned':c3,'C4_d200':c4},
                                   {'C1_PPMI':vocab,'C2_raw':rawv,'C3_cleaned':vocab,'C4_d200':vocab})
    return comparison

if __name__=='__main__':run()

In [ ]:
%%writefile src/annotation.py
"""Part 2 annotation: transparent rule-based silver labels, manually curated seeds.
Manually spot-check generated CoNLL files; no human-gold labels are fabricated.
"""
from __future__ import annotations
import re, random, json
from collections import Counter,defaultdict
from pathlib import Path
from .common import DATA, RES, tokenize, normalize, load_articles, stratified_split, show_counts, save_json, SENT_RE
from .a1_linguistics import urdu_stem, urdu_lemmatize

# The hand-curated root words below are intentionally readable and editable; the
# script adds common morphological variants to increase coverage, not pretrained NLP.
NOUN_ROOTS='''
city village country nation province district region street market house home room door window table chair school college university student teacher classroom book notebook pen pencil paper class lesson exam result degree education knowledge language word sentence story article news newspaper journalist report interview camera photograph video television radio internet website computer laptop phone mobile network signal software hardware application program algorithm model data dataset database file folder server system device machine robot science research study experiment theory method project assignment lecture library laboratory office company business industry factory shop store customer employee employer manager director president minister leader government parliament assembly election vote party policy law constitution court judge lawyer police army soldier officer border airport railway train bus car truck road bridge river canal lake sea mountain valley forest park garden tree flower fruit vegetable crop farmer agriculture wheat rice cotton sugar milk water food bread meat fish egg tea coffee breakfast lunch dinner restaurant hotel kitchen family father mother brother sister son daughter child children man woman person people friend neighbour community society culture tradition festival holiday health hospital clinic patient doctor nurse medicine vaccine disease infection treatment surgery blood heart brain body hand foot eye ear nose face hair skin bone muscle sport sports match team cricket football hockey tennis badminton player coach captain wicket ball bat stadium referee goal score tournament league champion championship win loss victory defeat economy bank money currency price tax trade commerce import export budget inflation salary wage income expense investment finance company market industry product service technology transport environment climate weather rain flood storm earthquake temperature energy power electricity gas oil petrol engine fuel security safety peace conflict war treaty diplomacy embassy country capital foreign affairs global world continent island geography religion mosque temple church school doctor lawyer engineer journalist writer poet artist actor singer film music poetry literature museum history'''.split()
VERB_ROOTS='''
be have do say go get make know think take see come want look use find give tell work call try ask need feel become leave put mean keep let begin seem help talk turn start show hear play run move live believe bring happen write provide sit stand lose pay meet include continue set learn change lead understand watch follow stop create speak read allow add spend grow open walk win offer remember love consider appear buy wait serve die send expect build stay fall cut reach kill remain suggest raise pass sell require report decide pull return explain hope develop carry break receive agree support hit produce eat cover catch draw choose cause point listen realize place close involve increase decrease reduce improve prepare train evaluate test compute encode decode classify label predict compare analyze plot save load print transform normalize initialize update optimize calculate backpropagate propagate transfer communicate infer inferencing generate finish publish discuss describe identify measure count check verify review revise edit submit download upload connect commit push run execute compile debug implement construct split sample select annotate assign score tag name align reconstruct process monitor inspect interpret learn teach study focus complete schedule plan organize evaluate compare determine record collect clean preprocess stem lemmatize tokenize filter map count select average summarize join merge repeat shuffle reset return forward backward backwarding mask pad truncate initialize tune freeze unfreeze inject attend attendable attendable translate translateable search retrieve extract compare correlate rank discover investigate observe state establish confirm distinguish consider conclude hypothesize model organize collaborate coordinate manage mediate authorize validate navigate draw calculate estimate measure compute multiply divide subtract add convert travel walk drive arrive depart visit call vote elect train qualify play bat bowl win lose score draw compete match serve treat heal vaccinate nurse diagnose fight negotiate import export manufacture trade purchase sell pay hire fire teach learn graduate educate grow harvest sow cook drink wash clean build design program deploy configure evaluate deploy connect pair compare summarize discuss report draft finalize write illustrate interpret visualize detect discover classify detect predict classify inspect select assign type annotate mask compute optimize test develop design improve reduce shorten accelerate accelerate achieve finish'''.split()
ADJ_ROOTS='''
good bad new old young large small big tiny huge long short wide narrow high low deep shallow tall brief fast slow quick rapid early late easy hard difficult simple complex clear unclear correct incorrect right wrong true false real fake authentic modern ancient current recent previous next first final last initial original official public private common rare frequent unusual unique normal special important significant minor major basic advanced powerful weak strong healthy sick dangerous safe secure insecure open closed active passive busy free poor rich wealthy expensive cheap affordable valuable economic financial political social cultural national international local foreign global domestic rural urban public private legal illegal medical clinical scientific technical digital virtual physical natural artificial human automated manual efficient inefficient accurate inaccurate precise vague robust fragile stable unstable fixed variable random deterministic fair unfair unbiased biased positive negative good better best poor poorer worst reliable unreliable responsible irresponsible ethical unethical intelligent smart creative innovative conventional traditional general specific particular detailed comprehensive incomplete complete helpful useful useless difficult challenging ambitious practical theoretical experimental analytical empirical semantic syntactic lexical grammatical contextual relational meaningful meaningless relevant irrelevant unrelated independent dependent interdependent visible invisible transparent opaque noisy clean messy dirty smooth rough soft hard light dark bright dull calm angry happy sad excited upset worried serious funny formal informal casual respectful polite friendly hostile supportive critical skeptical optimistic pessimistic objective subjective clear concise verbose brief long informative descriptive excellent average normal odd early late immediate delayed delayed available unavailable successful unsuccessful confident uncertain effective ineffective predictive generative realistic unrealistic strong weak strong patient impatient original copied human academic professional educational linguistic computational statistical neural hierarchical distributed sequential recurrent bidirectional unidirectional frozen trainable learned learnable generalizable accurate precise distinct different similar identical consistent inconsistent novel complex simple deep shallow human social government governmental urban rural regional federal provincial municipal parliamentary electoral economic industrial agricultural environmental geological geographical demographic historical historical ancient medieval religious medical biological chemical mathematical logical approximate exact observed measured reported tested valid invalid trained untrained significant small medium large black white green red blue yellow brown golden silver loud quiet bright dark rainy sunny cloudy windy foggy severe mild temporary permanent mobile stationary independent international healthy wealthy mature minor adult young elderly digital electronic mechanical visual textual spoken written conversational structured unstructured linked connected disconnected efficient trustworthy honest dishonest genuine fabricated'''.split()
ADV_WORDS='''quickly slowly carefully clearly correctly incorrectly fully partly often never always sometimes usually rarely frequently recently currently previously immediately finally early late here there now then today tomorrow yesterday very quite rather almost already still only also even especially extremely significantly strongly weakly approximately exactly relatively directly indirectly naturally manually automatically independently jointly closely differently similarly gradually suddenly rapidly safely openly honestly truly frankly mostly mainly generally specifically locally globally nationally internationally publicly privately actively passively confidently critically analytically practically easily hardly nicely badly well together apart ahead behind above below upward downward forward backward inside outside everywhere anywhere nowhere perhaps maybe probably certainly apparently likely surely absolutely strongly greatly rapidly simply directly respectively subsequently primarily secondarily automatically consistently independently thoroughly continuously steadily increasingly sharply slightly moderately nearly barely completely partly annually monthly weekly daily hourly sometimes ultimately therefore however accordingly additionally moreover consequently respectively even yet furthermore similarly differently clearly silently loudly exactly roughly gradually quickly slowly'''.split()
PRON_WORDS='''i me my mine you your yours he him his she her hers it its we us our ours they them their theirs who whom whose which what whoever myself yourself himself herself ourselves themselves mein main mujhe mera meri hum ham hamara tum tumhara aap aapka woh wo yeh ye un in kisi koi sab khud میں مجھے میرا میری ہم ہمارا تم تمہارا آپ وہ یہ ان اُن اسے انہیں کوئی کسی خود'''.split()
DET_WORDS='''a an the this that these those each every some any no few many much several all both either neither one another other such yeh wo ye woh is us iss uss ek koi kuch har tamam چند کئی کوئی یہ وہ اس ایک ایکہ'''.split()
CONJ_WORDS='''and or but nor yet so because although whereas while if unless when since after before until as whether neither either aur ya lekin magar kyunki agar jab tak پھر جبکہ اور یا لیکن مگر کیونکہ اگر جب تاکہ'''.split()
POST_WORDS='''in on at to from for with about under over above below between among within without during before after through across behind near beside into onto off up down by of per via than tak se ko mein main par pe ka ki ke kay kayliay saath baad pehle liye کے کی کا کو سے میں پر تک ساتھ بعد پہلے لیے'''.split()
PROPN_WORDS='''pakistan india karachi islamabad lahore peshawar quetta punjab sindh kpk balochistan asia china afghanistan pakistan india un islamabad karachi پنجاب سندھ پاکستان بھارت لاہور کراچی اسلام آباد پشاور کوئٹہ'''.split()

# The hand-crafted lexicon has >200 entries for each open class NOUN/VERB/ADJ
# after conservative surface expansions. The corpus may be Urdu script, for
# which the user must extend A1-specific lexicon before claiming quality.
# Manually selected Urdu vocabulary for the actual BBC Urdu script. Assignment 1
# removes Latin script, so an English-only lexicon would label nearly all tokens UNK.
UR_NOUNS="""پاکستان حکومت عدالت ریاست ملک شہر ضلع گاؤں صوبہ اسمبلی پارلیمان صدر وزیراعظم وزیر سیاسی جماعت پارٹی انتخاب ووٹ شہری عوام خاتون خواتین مرد بچہ بچے خاندان والد والدہ بیٹا بیٹی بھائی بہن والدین نوجوان بزرگ استاد طالبعلم طالبہ تعلیم سکول کالج یونیورسٹی امتحان کتاب کلاس سبق کورس تحقیق سائنس ٹیکنالوجی کمپیوٹر موبائل انٹرنیٹ مشین روبوٹ نظام توانائی بجلی گیس پانی بارش سیلاب موسم ہوا درجہ حرارت زلزلہ آفت نقصان امداد علاقے علاقہ گلی سڑک راستہ پولیس فوج سپاہی افسر شہری ہسپتال صحت ڈاکٹر مریض بیماری دوا علاج ویکسین وائرس طبی کارکن خبر رپورٹ صحافی ٹی وی اخبار میڈیا پروگرام ویڈیو تصویر ویب سائٹ دفتر ادارہ عدالت جج وکیل مقدمہ فیصلہ قانون جرم سزا حفاظت احتجاج مظاہرہ احتجاجی بجٹ معیشت تجارت کاروبار سرمایہ قیمت پیسہ روپے ڈالر ٹیکس بینک مارکیٹ صنعت زراعت فصل کسان گندم چاول سبزی گوشت روٹی سفر جہاز ٹرین ریلوے گاڑی بس موٹر سائیکل ٹریفک حادثہ کرکٹ میچ کھیل کھلاڑی ٹیم کپتان گیند بلے باز وکٹ رن فتح شکست ٹورنامنٹ فٹبال گول میدان اسٹیڈیم جنگ امن مذاکرات تنازع سرحد فوجی حکومتیں ممالک مذاکراتی معاہدہ دنیا خطہ براعظم امریکا بھارت چین افغانستان ایران سعودی عرب برطانیہ پاکستانیت ثقافت زبان فن فنکار فلم ڈرامہ گلوکار موسیقی تقریب جشن عید رمضان ریاضت کام روزگار ملازمت تنخواہ محنت خاندان معاشرہ آبادی مردم شماری تاریخ زمانہ مستقبل مسئلہ حل تبدیلی منصوبہ ترقی منصوبہ بندی نتیجہ سوال جواب وجہ اثر تحقیق خبرنامہ شعبہ تعلیمات ضروریات""".split()
UR_VERBS="""کر ہو بن جا آ چل دیکھ سن بول کہ لکھ پڑھ سیکھ سکھا بتا پوچھ سمجھ سوچ جان مان مانگ رکھ اٹھ بیٹھ دے لے لا بھیج بھیجوا پہنچ کھول بند شروع ختم روک چھوڑ بدل رہ اٹھا گرا ڈھونڈ نکال تلاش بنا بت بنا بڑھ کم سنبھال سنبھل شروع کروا ختم کروا سنبھال سنبھلو پہنچا واپس آ واپس جا مل پھیل بچ بچا مار قتل روک پکڑ گرفتار حملہ کر اطلاع دے اعلان کر جاری کر قبول کر رد کر نظر آ محسوس کر استعمال کر تیار کر جمع کر منتخب کر ہار جیت کھیل حصہ لے کام کر کام کرا مؤخر کر تجویز دے مانگ مطالبہ کر پیش کر تحقیق کر ثابت کر ظاہر کر طے کر بنا بس گزار چلا بچاؤ کر خبر دے پہنچا لکھوا جمع کروا یاد کر قبول کرا تبادلہ کر رابطہ کر ترقی کر سفر کر پڑھا سونپ متعارف کرا شروع ہو ختم ہو پڑھنے سیکھنے بولنے کرنے ہونے جانے آنے رکھنے لینے دینے چھپنے ماننے بڑھنے گرنے بڑھا کم ہوا گر گئے کر لیا کی گئی کہا گیا کہتی کہتا کرتا کرتے کرتا رہی رہا رہے موجود ہے ہیں تھا تھے ہوگی ہوگا گیا گئی گئے کریں کرے کرتا کرلیں کرو""".split()
UR_ADJS="""بڑا چھوٹا لمبا مختصر نیا پرانا اچھا برا بہتر بہترین مشکل آسان اہم ضروری قومی مقامی صوبائی وفاقی عالمی بین الاقوامی غیر ملکی ملکی پاکستانی بھارتی چینی امریکی افغان ایرانی سعودی برطانوی سرکاری نجی سیاسی معاشی اقتصادی تجارتی زرعی سماجی معاشرتی تعلیمی طبی صحت مند بیمار علمی جدید قدیم تاریخی موجودہ سابق آئندہ اگلا پچھلا حالیہ پہلا آخری دوسرا تیسرا کم زیادہ بہت فوری دیرینہ عارضی مستقل صاف گندا گرم ٹھنڈا سرد خشک گیلا طویل مضبوط کمزور تیز سست مہنگا سستا مستحکم غیر مستحکم کامیاب ناکام مناسب غیر مناسب مثبت منفی واضح مبہم سنجیدہ عام خاص حقیقی فرضی محتاط سخت نرم خوش غمگین غریب امیر نایاب مقبول معروف مشہور غیر معروف قابل ناممکن ممکن خوبصورت بدصورت دلچسپ قابل ذکر قابل اعتماد معتبر اہم ترین مختلف یکساں الگ مشترکہ شدید معمولی حساس محفوظ غیر محفوظ متاثرہ ہلاک زخمی زندہ طویل مدت قلیل مدت انسانی قدرتی موسمی ماحولیاتی معاشرتی انتظامی قانونی عدالتی مالی تکنیکی ڈیجیٹل سائنسی جدید ترین بنیادی اضافی اصلی غلط درست سادہ پیچیدہ ترقی یافتہ غیر ترقی یافتہ شمالی جنوبی مشرقی مغربی چھوٹے بڑے مضبوط تر آزاد مقید کھلا بند محفوظ ترین مؤثر موثر سرگرم غیر فعال آباد غیر آباد پرامن پُرامن صحت مند خطرناک نارمل معمول کا ذہین فعال بنیادی کم ترین زیادہ ترین""".split()
UR_ADVS="""اب آج کل پہلے بعد پھر فوری جلدی آہستہ تیزی سے یہاں وہاں کبھی اکثر ہمیشہ دوبارہ مسلسل یکدم بالکل تقریباً ممکنہ طور پر خاص طور پر عام طور پر معمولاً فی الحال رات دن اسی دوران اس دوران فوراً کبھی کبھار شاید یقیناً نمایاں طور پر عموماً زیادہ تر کم از کم اس لیے البتہ مزید صرف بہت بھی ابھی کل گزشتہ فوری طور پر""".split()


def build_lexicon():
    lex={}
    base=[('NOUN',NOUN_ROOTS),('VERB',VERB_ROOTS),('ADJ',ADJ_ROOTS),
          ('ADV',ADV_WORDS),('PRON',PRON_WORDS),('DET',DET_WORDS),
          ('CONJ',CONJ_WORDS),('POST',POST_WORDS),('PROPN',PROPN_WORDS)]
    for tag,words in base:
        for w in words:lex[normalize(w)]=tag
    for tag,words in [('NOUN',UR_NOUNS),('VERB',UR_VERBS),('ADJ',UR_ADJS),('ADV',UR_ADVS)]:
        for w in words:
            norm=normalize(w)
            lex[norm]=tag
            # A1 corpus already contains stemmed forms, so use its real rules.
            for candidate in (urdu_stem(norm),urdu_lemmatize(norm)):
                lex.setdefault(normalize(candidate),tag)
            # Transparent, programmatic inflectional expansions from curated Urdu roots.
            if ' ' not in norm:
                suffixes= ('وں','یں','ات') if tag=='NOUN' else ('نا','تا','تی','تے') if tag=='VERB' else ('ی','ے') if tag=='ADJ' else ()
                for aff in suffixes:lex.setdefault(norm+aff,tag)
    for root in NOUN_ROOTS:
        if re.fullmatch('[a-z]+',root):
            for v in [root+'s',root+'es']:lex.setdefault(v,'NOUN')
    for root in VERB_ROOTS:
        if re.fullmatch('[a-z]+',root):
            for v in [root+'s',root+'ed',root+'ing']:lex.setdefault(v,'VERB')
    for root in ADJ_ROOTS:
        if re.fullmatch('[a-z]+',root):
            for v in [root+'er',root+'est',root+'ly']:lex.setdefault(v,'ADJ' if not v.endswith('ly') else 'ADV')
    for w in [str(i) for i in range(1000)]:lex[w]='NUM'
    counts=Counter(lex.values())
    for tag in ['NOUN','VERB','ADJ']:
        if counts[tag]<200:raise RuntimeError(f'Lexicon size for {tag} is {counts[tag]}, expected >= 200')
    return lex,dict(counts)

# Seed gazetteer: 50+ Pakistani persons, 50+ locations and 30+ organizations.
PERSONS='''
Muhammad Ali Jinnah|Allama Iqbal|Liaquat Ali Khan|Fatima Jinnah|Benazir Bhutto|Zulfikar Ali Bhutto|Nawaz Sharif|Shehbaz Sharif|Imran Khan|Bilawal Bhutto|Asif Ali Zardari|Pervez Musharraf|Arif Alvi|Mamnoon Hussain|Yousaf Raza Gillani|Shahid Khaqan Abbasi|Raja Pervez Ashraf|Chaudhry Shujaat Hussain|Mian Muhammad Nawaz|Maryam Nawaz|Hamza Shehbaz|Khawaja Asif|Ishaq Dar|Shaukat Aziz|Abdul Sattar Edhi|Bilquis Edhi|Malala Yousafzai|Abdus Salam|Abdul Qadeer Khan|Dr Abdul Qadeer Khan|Javed Miandad|Wasim Akram|Waqar Younis|Shoaib Akhtar|Shahid Afridi|Babar Azam|Shaheen Afridi|Mohammad Rizwan|Sarfaraz Ahmed|Inzamam ul Haq|Younis Khan|Misbah ul Haq|Sana Mir|Nida Dar|Jahangir Khan|Jansher Khan|Arshad Nadeem|Naseem Shah|Fakhar Zaman|Shan Masood|Azhar Ali|Mohammad Amir|Hassan Ali|Sarfraz Ahmed|Atif Aslam|Nusrat Fateh Ali Khan|Abida Parveen|Ali Zafar|Fawad Khan|Mahira Khan|Saba Qamar|Hina Rabbani Khar|Sherry Rehman|Ahsan Iqbal|Miftah Ismail|Hafeez Shaikh|Rana Sanaullah|Murad Saeed|Asad Umar|Shaukat Tarin
'''.strip().split('|')
LOCATIONS='''
Pakistan|Islamabad|Karachi|Lahore|Rawalpindi|Peshawar|Quetta|Multan|Faisalabad|Hyderabad|Sukkur|Gujranwala|Sialkot|Bahawalpur|Sargodha|Abbottabad|Mansehra|Mardan|Swat|Mingora|Nowshera|Charsadda|Kohat|Bannu|Dera Ismail Khan|Dera Ghazi Khan|Rahim Yar Khan|Sahiwal|Okara|Kasur|Sheikhupura|Jhelum|Chakwal|Attock|Taxila|Wah Cantt|Murree|Muzaffarabad|Gilgit|Skardu|Hunza|Chitral|Dir|Kalam|Kaghan|Naran|Kashmir|Azad Kashmir|Punjab|Sindh|Balochistan|Khyber Pakhtunkhwa|Gwadar|Turbat|Khuzdar|Larkana|Nawabshah|Thatta|Tharparkar|Badin|Mirpur Khas|Rohri|Kotri|Layyah|Mianwali|Bhakkar|Hafizabad|Narowal|Vehari|Lodhran|Jhang|Toba Tek Singh|Chiniot|Pakpattan|Haripur|Hangu|Kurram|Kohistan|Upper Dir|Lower Dir|Bajaur|Khyber|Mohmand
'''.strip().split('|')
ORGS='''
National Assembly|Senate of Pakistan|Supreme Court|Supreme Court of Pakistan|High Court|Islamabad High Court|Lahore High Court|Sindh High Court|Peshawar High Court|Federal Shariat Court|Election Commission|Election Commission of Pakistan|Pakistan Army|Pakistan Air Force|Pakistan Navy|Government of Pakistan|State Bank of Pakistan|National Bank of Pakistan|United Bank Limited|Habib Bank Limited|Pakistan International Airlines|Pakistan Railways|Pakistan Cricket Board|PCB|Pakistan Super League|PSL|Higher Education Commission|HEC|National Database Registration Authority|NADRA|National Disaster Management Authority|NDMA|Federal Board of Revenue|FBR|Ministry of Finance|Ministry of Education|Ministry of Health|Ministry of Foreign Affairs|Ministry of Interior|Ministry of Defence|World Health Organization|United Nations|UNICEF|UNESCO|World Bank|International Monetary Fund|IMF|FAST NUCES|Quaid i Azam University|University of Karachi|National University of Sciences and Technology|NUST|University of Punjab|Lahore University of Management Sciences|LUMS|Pakistan Tehreek e Insaf|Pakistan Muslim League N|Pakistan Peoples Party|Dawn News|Geo News|ARY News|Radio Pakistan|Express Tribune|Edhi Foundation|Shaukat Khanum Hospital|Aga Khan University Hospital
'''.strip().split('|')
MISC='''
Eid ul Fitr|Eid ul Adha|Ramadan|Independence Day|Pakistan Day|Urdu|Islam|World Cup|Cricket World Cup|COVID 19|Corona Virus|Kashmir Day
'''.strip().split('|')

# Urdu-script seeds are necessary because cleaned.txt removes Roman/English text.

# Each proper name is a phrase; use a pipe delimiter, not a whitespace split.
PERSONS_UR='''محمد علی جناح|علامہ اقبال|لیاقت علی خان|فاطمہ جناح|بے نظیر بھٹو|ذوالفقار علی بھٹو|نواز شریف|شہباز شریف|عمران خان|بلاول بھٹو|آصف علی زرداری|پرویز مشرف|عارف علوی|ممنون حسین|یوسف رضا گیلانی|شاہد خاقان عباسی|راجہ پرویز اشرف|چوہدری شجاعت حسین|مریم نواز|حمزہ شہباز|خواجہ آصف|اسحاق ڈار|شوکت عزیز|عبدالستار ایدھی|بلقیس ایدھی|ملالہ یوسفزئی|عبدالسلام|عبدالقدیر خان|جاوید میانداد|وسیم اکرم|وقار یونس|شعیب اختر|شاہد آفریدی|بابر اعظم|شاہین آفریدی|محمد رضوان|سرفراز احمد|انضمام الحق|یونس خان|مصباح الحق|ثنا میر|ندا ڈار|جہانگیر خان|جان شیر خان|ارشد ندیم|نسیم شاہ|فخر زمان|شان مسعود|اظہر علی|محمد عامر|حسن علی|عاطف اسلم|نصرت فتح علی خان|عابدہ پروین|علی ظفر|فواد خان|ماہرہ خان|صبا قمر|حنا ربانی کھر|شیری رحمان|احسن اقبال|مفتاح اسماعیل|حفیظ شیخ|رانا ثنااللہ|مراد سعید|اسد عمر|شوکت ترین'''.split('|')
LOCATIONS_UR='''پاکستان|اسلام آباد|کراچی|لاہور|راولپنڈی|پشاور|کوئٹہ|ملتان|فیصل آباد|حیدر آباد|سکھر|گوجرانوالہ|سیالکوٹ|بہاولپور|سرگودھا|ایبٹ آباد|مانسہرہ|مردان|سوات|مینگورہ|نوشہرہ|چارسدہ|کوہاٹ|بنوں|ڈیرہ اسماعیل خان|ڈیرہ غازی خان|رحیم یار خان|ساہیوال|اوکاڑہ|قصور|شیخوپورہ|جہلم|چکوال|اٹک|ٹیکسلا|واہ کینٹ|مری|مظفر آباد|گلگت|سکردو|ہنزہ|چترال|دیر|کالام|کاغان|ناران|کشمیر|آزاد کشمیر|پنجاب|سندھ|بلوچستان|خیبر پختونخوا|گوادر|تربت|خضدار|لاڑکانہ|نواب شاہ|ٹھٹہ|تھرپارکر|بدین|میرپور خاص|روہڑی|کوٹری|میانوالی|بھکر|حافظ آباد|نارووال|وہاڑی|لودھراں|جھنگ|چنیوٹ|پاکپتن|ہری پور|ہنگو|کرم|باجوڑ|خیبر|مہمند'''.split('|')
ORGS_UR='''قومی اسمبلی|سینیٹ آف پاکستان|سپریم کورٹ|سپریم کورٹ آف پاکستان|ہائی کورٹ|اسلام آباد ہائی کورٹ|لاہور ہائی کورٹ|سندھ ہائی کورٹ|پشاور ہائی کورٹ|وفاقی شرعی عدالت|الیکشن کمیشن|الیکشن کمیشن آف پاکستان|پاکستان آرمی|پاکستان فضائیہ|پاکستان بحریہ|حکومت پاکستان|اسٹیٹ بینک آف پاکستان|نیشنل بینک آف پاکستان|یونائیٹڈ بینک|حبیب بینک|پاکستان انٹرنیشنل ایئرلائنز|پاکستان ریلوے|پاکستان کرکٹ بورڈ|پاکستان سپر لیگ|ہائر ایجوکیشن کمیشن|نادرا|نیشنل ڈیزاسٹر مینجمنٹ اتھارٹی|ایف بی آر|وزارت خزانہ|وزارت تعلیم|وزارت صحت|وزارت خارجہ|وزارت داخلہ|وزارت دفاع|عالمی ادارہ صحت|اقوام متحدہ|یونیسف|یونیسکو|عالمی بینک|آئی ایم ایف|فاسٹ یونیورسٹی|قائد اعظم یونیورسٹی|جامعہ کراچی|نسٹ|پنجاب یونیورسٹی|لمز|پاکستان تحریک انصاف|پاکستان مسلم لیگ ن|پاکستان پیپلز پارٹی|ڈان نیوز|جیو نیوز|اے آر وائی نیوز|ریڈیو پاکستان|ایکسپریس نیوز|ایدھی فاؤنڈیشن|شوکت خانم ہسپتال|آغا خان یونیورسٹی ہسپتال'''.split('|')


def gazetteer():
    """Exact and conservative stem-normalized matches for A1-cleaned Urdu.

    Keep multiword entities intact. Do not overwrite ambiguous collisions.
    Labels remain weak/silver labels, not manually verified ground truth.
    """
    assert len(PERSONS)>=50 and len(LOCATIONS)>=50 and len(ORGS)>=30
    out={}; ambiguous=set()
    def add(key, typ):
        if not key or key in ambiguous:return
        if key in out and out[key]!=typ:
            ambiguous.add(key);out.pop(key,None)
        else:out[key]=typ
    for typ,items in [('PER',PERSONS+PERSONS_UR),('LOC',LOCATIONS+LOCATIONS_UR),
                      ('ORG',ORGS+ORGS_UR),('MISC',MISC)]:
        for item in items:
            seq=tuple(normalize(x) for x in tokenize(item))
            add(seq,typ)
            # Assignment 1 clean_and_process_article stems words individually.
            add(tuple(normalize(urdu_stem(w)) for w in seq),typ)
            add(tuple(normalize(urdu_lemmatize(w)) for w in seq),typ)
    return out

def ner_bio(tokens, gz):
    tags=['O']*len(tokens)
    max_len=max(map(len,gz))
    i=0
    while i<len(tokens):
        for end in range(min(len(tokens),i+max_len),i,-1):
            cat=gz.get(tuple(tokens[i:end]))
            if cat:
                tags[i]='B-'+cat
                for j in range(i+1,end):tags[j]='I-'+cat
                i=end;break
        else:i+=1
    return tags


POS_TAGS=['NOUN','VERB','ADJ','ADV','PRON','DET','CONJ','POST','NUM','PUNC','UNK','PROPN']
NER_TAGS=['O','B-PER','I-PER','B-LOC','I-LOC','B-ORG','I-ORG','B-MISC','I-MISC']


def rule_tag(w,lex):
    if re.fullmatch(r'[^\w\s]+',w):return 'PUNC'
    if w.lower()=='<num>' or re.fullmatch(r'[\d.,]+',w):return 'NUM'
    norm=normalize(w)
    if norm in lex:return lex[norm]
    # Assignment 1's own lexicon/morphology provides an interpretable fallback.
    # The original cleaned.txt has already been stemmed: this is only a lookup aid.
    for candidate in (urdu_lemmatize(norm), urdu_stem(norm),
                      urdu_stem(urdu_lemmatize(norm))):
        if candidate in lex:return lex[candidate]
    # Remaining English morphological rules are for raw/Roman tokens, if present.
    if re.fullmatch('[a-z]+',norm):
        for suffix in ['ing','ed','es','s']:
            if norm.endswith(suffix) and len(norm)>len(suffix)+2:
                stem=norm[:-len(suffix)]
                if stem in lex and lex[stem] in ['NOUN','VERB']:return lex[stem]
        if norm.endswith('ly'):return 'ADV'
        if norm.endswith(('tion','ment','ness','ity')):return 'NOUN'
        if norm.endswith(('ous','ive','able','al')):return 'ADJ'
    return 'UNK'


def create_sentences(articles):
    out=[]
    for art in articles:
        pieces=[s.strip() for s in SENT_RE.split(art['text']) if s.strip()]
        for s in pieces:
            tokens=tokenize(s,keep_punc=True)
            if 5<=len(tokens)<=120:
                out.append({'tokens':tokens,'topic':art['topic'],'article_id':art['id']})
    return out


def write_conll(samples,split,task):
    path=DATA/f'{task}_{split}.conll'
    with path.open('w',encoding='utf-8') as f:
        for sample in samples:
            for tok,tag in zip(sample['tokens'],sample[task]): f.write(f'{tok}\t{tag}\n')
            f.write('\n')


def run():
    """Exactly 500 varied sentences; ≥100 from three topics; entity-aware sample.

    Sampling uses gazetteer hits only, never model predictions. Splits are
    made before training-only oversampling; validation/test are not resampled.
    """
    random.seed(42)
    articles=load_articles();pool=create_sentences(articles)
    lex,lc=build_lexicon();gz=gazetteer()
    for s in pool:
        s['pos']=[rule_tag(w,lex) for w in s['tokens']]
        s['ner']=ner_bio([normalize(w) for w in s['tokens']],gz)
        s['entity_count']=sum(tag.startswith('B-') for tag in s['ner'])
    grouped=defaultdict(list)
    for s in pool:grouped[s['topic']].append(s)
    eligible=[t for t,ss in grouped.items() if len(ss)>=100]
    if len(eligible)<3:raise ValueError('Three topics with >=100 sentences required')
    rng=random.Random(42);selected=[];chosen=set()
    def choose(group,count):
        available=[s for s in group if id(s) not in chosen]
        positive=[s for s in available if s['entity_count']>0]
        # ~70% entity-containing sentences, rest random background examples.
        target=min(len(positive),int(count*.70))
        pick=rng.sample(positive,target) if target else []
        used={id(s) for s in pick}
        others=[s for s in available if id(s) not in used]
        if len(others)<count-target:raise ValueError('Insufficient distinct sentences')
        pick+=rng.sample(others,count-target)
        chosen.update(id(s) for s in pick)
        return pick
    major=sorted(eligible,key=lambda t:len(grouped[t]),reverse=True)[:3]
    for t in major:selected+=choose(grouped[t],100)
    selected+=choose(pool,200)
    rng.shuffle(selected)
    assert len(selected)==500 and len(chosen)==500
    labels=[s['topic'] for s in selected]
    tr,va,te=stratified_split(selected,labels,42)
    # Avoid accidentally having no entities in a holdout; no test data used
    # for learning or hyperparameter search.
    for split,ids in [('train',tr),('val',va),('test',te)]:
        chunk=[selected[i] for i in ids]
        write_conll(chunk,split,'pos');write_conll(chunk,split,'ner')
        print('POS',split,dict(Counter(t for s in chunk for t in s['pos'])))
        print('NER',split,dict(Counter(t for s in chunk for t in s['ner'])))
        print('NER entity spans',split,sum(s['entity_count'] for s in chunk))
    audits=[{'tokens':s['tokens'],'POS':s['pos'],'NER':s['ner'],
             'topic':s['topic'],'entity_count':s['entity_count']} for s in selected[:30]]
    save_json('annotation_review_samples.json',audits)
    summary={'sentences':500,'split_sizes':{k:len(v) for k,v in [('train',tr),('val',va),('test',te)]},
             'topics':dict(Counter(labels)),'lexicon_counts':lc,'gazetteer_unique_entries':len(gz),
             'entity_spans_by_split':{k:sum(selected[i]['entity_count'] for i in ids)
                 for k,ids in [('train',tr),('val',va),('test',te)]},
             'gazetteer_seed_counts':{'PER':len(PERSONS)+len(PERSONS_UR),
                 'LOC':len(LOCATIONS)+len(LOCATIONS_UR),'ORG':len(ORGS)+len(ORGS_UR),'MISC':len(MISC)},
             'label_warning':'Weak gazetteer labels; manual expert annotation still required for gold accuracy.',
             'sampling_note':'70% of each selection stratum selected by gazetteer-hit status; no augmentation after split.',
             'pos_tag_note':'PDF says 12 POS tags but lists 11; PROPN included as 12th.'}
    save_json('annotation_summary.json',summary)
    return summary

if __name__=='__main__':run()


In [ ]:
%%writefile src/conlleval.py
"""Minimal CoNLL-style entity-span scorer for BIO tag sequences.

Reads token gold predicted columns (blank lines delimit sentences) as written by
src.sequence. Exact matching of typed spans, including BIO boundary repair for I-X.
This is an independently implemented evaluator, not the original Perl conlleval.

Usage: python -m src.conlleval results/ner_conlleval_input.conll
"""
import sys, json
from .sequence import conlleval_scores


def read(path):
    gold=[];pred=[];g=[];p=[]
    for line in open(path,encoding='utf-8'):
        if not line.strip():
            if g:gold.append(g);pred.append(p);g=[];p=[]
            continue
        parts=line.strip().split()
        if len(parts)<3:raise ValueError('Expected token gold pred: '+line)
        g.append(parts[-2]);p.append(parts[-1])
    if g:gold.append(g);pred.append(p)
    return gold,pred


def evaluate(path):
    gold,pred=read(path)
    return conlleval_scores(gold,pred)[0]

if __name__=='__main__':
    data=evaluate(sys.argv[1])
    for typ,values in data.items():
        print(f'{typ:8s} P={values["precision"]*100:6.2f}% R={values["recall"]*100:6.2f}% F1={values["f1"]*100:6.2f}% '
              f'TP={values["TP"]} FP={values["FP"]} FN={values["FN"]}')

In [ ]:
%%writefile src/sequence.py
"""Part 2: from-scratch BiLSTM sequence labelling, CRF/Viterbi and ablations."""
from __future__ import annotations
import copy, json, time
from collections import Counter,defaultdict
from pathlib import Path
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence,pad_packed_sequence
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from sklearn.metrics import accuracy_score,f1_score,confusion_matrix
from .common import DATA, EMB, MODELS, FIGS, RES, seed_all, device, plot_curves,plot_confusion,save_json
from .annotation import POS_TAGS,NER_TAGS


class TaggedDataset(Dataset):
    def __init__(self,split,task,vocab):
        path=DATA/f'{task}_{split}.conll'
        if not path.exists():raise FileNotFoundError(f'{path} missing: run src.annotation first')
        lines=path.read_text('utf-8').splitlines();self.rows=[];one=[]
        for line in lines+['']:
            if line.strip():
                parts=line.split('\t')
                if len(parts)!=2:raise ValueError(f'Bad CoNLL line: {line}')
                one.append(parts)
            elif one:
                self.rows.append(([w for w,t in one],[t for w,t in one]));one=[]
        self.vocab=vocab; self.tag2id={t:i for i,t in enumerate(POS_TAGS if task=='pos' else NER_TAGS)}
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        ts,tags=self.rows[i]
        return ([self.vocab.get(t.lower(),1) for t in ts], [self.tag2id[t] for t in tags],ts)


def collate(batch):
    seqs,tags,texts=zip(*batch);lengths=torch.tensor([len(s) for s in seqs],dtype=torch.long)
    maxlen=int(lengths.max());ids=torch.zeros(len(batch),maxlen,dtype=torch.long)
    yy=torch.zeros(len(batch),maxlen,dtype=torch.long)
    mask=torch.arange(maxlen)[None,:]<lengths[:,None]
    for i,(x,y) in enumerate(zip(seqs,tags)):
        ids[i,:len(x)]=torch.tensor(x); yy[i,:len(y)]=torch.tensor(y)
    return ids,yy,mask,lengths,list(texts)


class LinearChainCRF(nn.Module):
    """Learned start/end/tag transitions. Exact forward normalizer and Viterbi MAP."""
    def __init__(self,n):
        super().__init__();self.n=n
        self.transitions=nn.Parameter(torch.empty(n,n))  # [previous,current]
        self.start=nn.Parameter(torch.empty(n));self.end=nn.Parameter(torch.empty(n))
        nn.init.uniform_(self.transitions,-.1,.1)
        nn.init.uniform_(self.start,-.1,.1);nn.init.uniform_(self.end,-.1,.1)
    def log_partition(self,e,mask):
        score=self.start+e[:,0]
        for t in range(1,e.size(1)):
            nxt=torch.logsumexp(score[:,:,None]+self.transitions[None,:,:]+e[:,t,None,:],dim=1)
            score=torch.where(mask[:,t,None],nxt,score)
        return torch.logsumexp(score+self.end,dim=1)
    def gold_score(self,e,y,mask):
        B,T=y.shape
        score=self.start[y[:,0]]+e[:,0,:].gather(1,y[:,0,None]).squeeze(1)
        for t in range(1,T):
            emit=e[:,t,:].gather(1,y[:,t,None]).squeeze(1)
            link=self.transitions[y[:,t-1],y[:,t]]
            score+=(emit+link)*mask[:,t]
        lengths=mask.long().sum(1)-1
        last=y.gather(1,lengths[:,None]).squeeze(1)
        return score+self.end[last]
    def forward(self,emissions,tags,mask):
        return (self.log_partition(emissions,mask)-self.gold_score(emissions,tags,mask)).mean()
    def viterbi(self,e,mask):
        score=self.start+e[:,0]
        bps=[]
        for t in range(1,e.size(1)):
            poss=score[:,:,None]+self.transitions[None,:,:]
            best,back=poss.max(dim=1)
            nxt=best+e[:,t]
            score=torch.where(mask[:,t,None],nxt,score)
            bps.append(back)
        score=score+self.end
        seqs=[]
        for i,L in enumerate(mask.sum(1).tolist()):
            cur=int(score[i].argmax());path=[cur]
            for t in range(int(L)-2,-1,-1):
                cur=int(bps[t][i,cur]);path.append(cur)
            seqs.append(path[::-1])
        return seqs


class BiLSTMTagger(nn.Module):
    def __init__(self,vocab_size,num_tags,weights,freeze=False,bidirectional=True,
                 dropout=.5,random_init=False,use_crf=False):
        super().__init__();d=int(weights.shape[1]);hidden=64
        self.embedding=nn.Embedding(vocab_size,d,padding_idx=0)
        self.initial_embedding_stats={"source":"random" if random_init else "C3 Word2Vec"}
        if not random_init:
            pretrained=torch.from_numpy(weights.copy()).float()
            # Skip-gram vectors start much smaller than nn.Embedding's N(0,1)
            # weights. Match initial feature scale without replacing or
            # re-training the learned C3 vectors. Use one global factor, so
            # semantic directions and relative pairwise magnitudes are retained.
            if use_crf:
                valid=pretrained[2:]  # ignore PAD and UNK for scale estimate
                original_std=float(valid.std(unbiased=False).item()) if valid.numel() else 0.0
                if original_std < 1e-8:raise ValueError('C3 vectors are degenerate; inspect embeddings_w2v.npy')
                factor=min(100.0,1.0/original_std)
                pretrained.mul_(factor)
                self.initial_embedding_stats.update(raw_std=original_std,
                                                    global_scale=factor,
                                                    scaled_std=float(pretrained[2:].std(unbiased=False)))
            self.embedding.weight.data.copy_(pretrained)
        # Ensure padding stays zero for BOTH pretrained and random ablation.
        with torch.no_grad():self.embedding.weight[0].zero_()
        self.embedding.weight.requires_grad_(not freeze)
        self.dropout=nn.Dropout(dropout)
        self.lstm=nn.LSTM(d,hidden,num_layers=2,batch_first=True,
                          bidirectional=bidirectional,dropout=dropout)
        self.classifier=nn.Linear(hidden*(2 if bidirectional else 1),num_tags)
        self.crf=LinearChainCRF(num_tags) if use_crf else None
    def emissions(self,x,lengths):
        e=self.dropout(self.embedding(x))
        packed=pack_padded_sequence(e,lengths.cpu(),batch_first=True,enforce_sorted=False)
        encoded,_=self.lstm(packed)
        out,_=pad_packed_sequence(encoded,batch_first=True,total_length=x.shape[1])
        return self.classifier(self.dropout(out))
    def loss(self,emit,y,mask):
        if self.crf is not None:return self.crf(emit,y,mask)
        return F.cross_entropy(emit.reshape(-1,emit.shape[-1])[mask.reshape(-1)],y.reshape(-1)[mask.reshape(-1)])
    def predict(self,emit,mask):
        if self.crf is not None:return self.crf.viterbi(emit,mask)
        return [row[:int(L)].tolist() for row,L in zip(emit.argmax(-1),mask.sum(-1))]


def get_spans(tags):
    """Conlleval BIO chunk boundaries, treating illegal I-X as a new B-X."""
    chunks=[];start=None;typ=None
    for i,tag in enumerate(list(tags)+['O']):
        prefix,kind=(tag.split('-',1)+[None])[:2] if '-' in tag else (tag,None)
        if typ is not None and (prefix in ('O','B') or kind!=typ):
            chunks.append((typ,start,i));start=None;typ=None
        if prefix in ('B','I') and typ is None:
            start=i;typ=kind
    return set(chunks)


def conlleval_scores(true_seq,pred_seq):
    """Strict span-level CoNLL evaluation with matching type and token boundaries.
    Also writes results/conlleval_input.conll for optional external conlleval check.
    """
    tags=['PER','LOC','ORG','MISC'];stats={t:{'TP':0,'FP':0,'FN':0} for t in tags}
    fps=[];fns=[]
    for idx,(gold,pred) in enumerate(zip(true_seq,pred_seq)):
        ga=get_spans(gold);pa=get_spans(pred)
        for t,a,b in (pa & ga):stats[t]['TP']+=1
        for t,a,b in (pa-ga):stats[t]['FP']+=1;fps.append({'sentence_index':idx,'span':[a,b],'type':t,'error':'false_positive'})
        for t,a,b in (ga-pa):stats[t]['FN']+=1;fns.append({'sentence_index':idx,'span':[a,b],'type':t,'error':'false_negative'})
    output={}
    for t in tags+['overall']:
        counts=stats[t] if t!='overall' else {k:sum(v[k] for v in stats.values()) for k in ['TP','FP','FN']}
        tp,fp,fn=counts['TP'],counts['FP'],counts['FN']
        precision=tp/(tp+fp) if tp+fp else 0.
        recall=tp/(tp+fn) if tp+fn else 0.
        f1=2*precision*recall/(precision+recall) if precision+recall else 0.
        output[t]={**counts,'precision':precision,'recall':recall,'f1':f1}
    return output,fps,fns


def infer_dataset(model,loader,task,dev):
    model.eval();all_gold=[];all_pred=[];all_tokens=[];totalloss=0.;num=0
    tags=POS_TAGS if task=='pos' else NER_TAGS
    with torch.no_grad():
        for x,y,m,l,tokens in loader:
            x=x.to(dev);y=y.to(dev);m=m.to(dev)
            emit=model.emissions(x,l);totalloss+=float(model.loss(emit,y,m).item());num+=1
            pred=model.predict(emit,m)
            gold=[row[:int(L)].tolist() for row,L in zip(y,l)]
            all_gold.extend([[tags[i] for i in row] for row in gold])
            all_pred.extend([[tags[i] for i in row] for row in pred]);all_tokens.extend(tokens)
    if task=='pos':
        flat_g=[t for row in all_gold for t in row]; flat_p=[t for row in all_pred for t in row]
        metrics={'accuracy':float(accuracy_score(flat_g,flat_p)),
                 'macro_f1':float(f1_score(flat_g,flat_p,labels=POS_TAGS,average='macro',zero_division=0))}
        primary=metrics['macro_f1']
    else:
        metrics,_,_=conlleval_scores(all_gold,all_pred);primary=metrics['overall']['f1']
    return {'loss':totalloss/max(1,num),'primary_f1':primary,'metrics':metrics,
            'gold':all_gold,'pred':all_pred,'tokens':all_tokens}


def train_one(task,name,vocab,weights,freeze=False,bidir=True,dropout=.5,random_init=False,
              use_crf=False,max_epochs=30,batch_size=24):
    seed_all(42);dev=device()
    datasets={k:TaggedDataset(k,task,vocab) for k in ['train','val','test']}
    loaders={k:DataLoader(v,batch_size=batch_size,shuffle=(k=='train'),collate_fn=collate) for k,v in datasets.items()}
    if task=='ner':
        # Rebalance only TRAINING batches, not validation/test references.
        # Preserve whole BIO sentences and avoid generating synthetic labels.
        sentence_weights=[]
        for _,tags in datasets['train'].rows:
            spans=sum(x.startswith('B-') for x in tags)
            sentence_weights.append(min(8.0,1.0+4.0*spans))
        sampler=WeightedRandomSampler(sentence_weights,len(sentence_weights),replacement=True,
                                      generator=torch.Generator().manual_seed(42))
        loaders['train']=DataLoader(datasets['train'],batch_size=batch_size,
                                    sampler=sampler,collate_fn=collate)
    tags=POS_TAGS if task=='pos' else NER_TAGS
    model=BiLSTMTagger(len(vocab),len(tags),weights,freeze,bidir,dropout,random_init,use_crf).to(dev)
    opt=torch.optim.Adam(filter(lambda p:p.requires_grad,model.parameters()),lr=.001,weight_decay=.0001)
    class_weights=None
    if task=='ner':
        # Compute token weights from TRAIN annotations only; do not look at val/test.
        counts=Counter(t for _,sentence_tags in datasets['train'].rows for t in sentence_tags)
        background=max(1,counts['O'])
        vals=[1.0 if t=='O' else min(8.0,max(2.0,(background/max(1,counts[t]))**.5))
              for t in tags]
        class_weights=torch.tensor(vals,dtype=torch.float32,device=dev)
        print(name,'TRAIN-only NER class weights:',dict(zip(tags,[round(v,2) for v in vals])))
        print(name,'embedding initialization:',model.initial_embedding_stats)
    # Patience remains five validation-F1 non-improvements, but entity models
    # receive a minimum warm-up: their F1 is often zero in the early epochs.
    minimum_ner_epochs=12
    hist=[];best=-1.;bad=0;beststate=None;last_epoch=0
    for epoch in range(1,max_epochs+1):
        t0=time.perf_counter();model.train();sum_loss=0.;steps=0
        for x,y,m,l,_ in loaders['train']:
            x=x.to(dev);y=y.to(dev);m=m.to(dev)
            opt.zero_grad(set_to_none=True);em=model.emissions(x,l)
            loss=model.loss(em,y,m)
            if task=='ner':
                # Supplemental supervision combats the CRF's all-O collapse.
                # CRF + Viterbi remain the primary structured model and decoder.
                weighted_ce=F.cross_entropy(em[m],y[m],weight=class_weights)
                loss=(loss + 0.75*weighted_ce) if use_crf else weighted_ce
            loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=5.)
            opt.step();sum_loss+=float(loss.item());steps+=1
        va=infer_dataset(model,loaders['val'],task,dev)
        row={'epoch':epoch,'train_loss':sum_loss/max(steps,1),'val_loss':va['loss'],
             'val_f1':va['primary_f1'],'epoch_seconds':round(time.perf_counter()-t0,3)}
        hist.append(row);print(name,'epoch',epoch,'trainloss',round(row['train_loss'],4),'valF1',round(row['val_f1'],4))
        if va['primary_f1']>best+1e-7:
            best=va['primary_f1'];bad=0;beststate=copy.deepcopy(model.state_dict());last_epoch=epoch
        elif task!='ner' or epoch>minimum_ner_epochs:
            bad+=1
            if bad>=5:break
        # Do not count zero-F1 warm-up epochs against the five-epoch patience.
        if task=='ner' and epoch==minimum_ner_epochs:bad=0
    model.load_state_dict(beststate)
    test=infer_dataset(model,loaders['test'],task,dev)
    torch.save({'state_dict':model.state_dict(),'task':task,'tags':tags,'vocab_size':len(vocab),
                'hyperparams':{'freeze':freeze,'bidirectional':bidir,'dropout':dropout,
                               'random_init':random_init,'crf':use_crf},'best_epoch':last_epoch},MODELS/f'{name}.pt')
    plot_curves(hist,['train_loss','val_loss'],f'{name}_loss.png',f'{name}: learning curves')
    plot_curves(hist,['val_f1'],f'{name}_validation_f1.png',f'{name}: validation F1')
    result={'name':name,'task':task,'best_epoch':last_epoch,'epochs_run':len(hist),
            'best_validation_f1':best,'test_f1':test['primary_f1'],
            'test_metrics':test['metrics'],'training_seconds_per_epoch':round(float(np.mean([x['epoch_seconds'] for x in hist])),2),
            'history':hist}
    save_json(f'{name}_metrics.json',result)
    return model,loaders,test,result


def evaluate_pos(test):
    gs=[x for seq in test['gold'] for x in seq];ps=[x for seq in test['pred'] for x in seq]
    cm=confusion_matrix(gs,ps,labels=POS_TAGS)
    plot_confusion(cm,POS_TAGS,'pos_confusion.png','POS tagging: 12-tag confusion matrix')
    most=[]
    for i,g in enumerate(POS_TAGS):
        for j,p in enumerate(POS_TAGS):
            if i!=j and cm[i,j]>0:
                examples=[]
                for toks,gseq,pseq in zip(test['tokens'],test['gold'],test['pred']):
                    wrong=[k for k,(a,b) in enumerate(zip(gseq,pseq)) if a==g and b==p]
                    if wrong and len(examples)<2:
                        examples.append({'sentence':' '.join(toks),'mismatched_token':toks[wrong[0]],'gold':g,'predicted':p})
                most.append({'gold':g,'predicted':p,'confusions':int(cm[i,j]),'examples':examples})
    most=sorted(most,key=lambda x:-x['confusions'])[:3]
    save_json('pos_confused_tag_pairs.json',most)
    print('Top 3 POS confusions:',most)
    return most


def evaluate_ner(test):
    scores,fp,fn=conlleval_scores(test['gold'],test['pred'])
    def describe(errors):
        result=[]
        for e in errors[:5]:
            i=e['sentence_index'];a,b=e['span'];tokens=test['tokens'][i]
            gold_seq=test['gold'][i];pred_seq=test['pred'][i]
            gold_here=gold_seq[a:b];pred_here=pred_seq[a:b]
            expected=set(x.replace('B-','').replace('I-','') for x in gold_here if x!='O')
            predicted=set(x.replace('B-','').replace('I-','') for x in pred_here if x!='O')
            if e['error']=='false_positive':
                if expected and expected!=predicted:
                    why=f'Wrong entity type: gold spans show {sorted(expected)}, prediction shows {sorted(predicted)}.'
                elif expected:
                    why='Entity type overlaps, but predicted start/end boundary does not match the gold span.'
                else:
                    why='Model predicted an entity where the silver-label gazetteer marked no entity.'
            else:
                if predicted and expected!=predicted:
                    why=f'Model used the wrong entity type: expected {sorted(expected)}, predicted {sorted(predicted)}.'
                elif predicted:
                    why='Model predicted an entity but did not recover the complete reference span.'
                else:
                    why='Model missed a reference entity entirely, potentially due to lexical/contextual sparsity.'
            result.append({**e,'entity':' '.join(tokens[a:b]),'sentence':' '.join(tokens),
                           'gold_tags':gold_here,'predicted_tags':pred_here,'explanation':why})
        return result
    errors={'false_positives':describe(fp),'false_negatives':describe(fn),
            'error_warning':'Reference tags are generated automatically, not human-annotated gold.'}
    save_json('ner_error_analysis.json',errors)
    # conlleval compatible input file: token gold pred, one blank line per sentence.
    with (RES/'ner_conlleval_input.conll').open('w',encoding='utf-8') as f:
        for tok,gold,pred in zip(test['tokens'],test['gold'],test['pred']):
            for x,y,z in zip(tok,gold,pred):f.write(f'{x} {y} {z}\n')
            f.write('\n')
    # Re-read the actual CoNLL triplets through our independent conlleval-style CLI parser.
    from .conlleval import evaluate
    cli_scores=evaluate(str(RES/'ner_conlleval_input.conll'))
    assert abs(cli_scores['overall']['f1']-scores['overall']['f1'])<1e-10
    save_json('ner_conlleval_span_scores.json',cli_scores)
    with (RES/'ner_conlleval_metrics.txt').open('w',encoding='utf-8') as f:
        for typ,metrics in cli_scores.items():
            f.write(f'{typ} P={metrics["precision"]:.5f} R={metrics["recall"]:.5f} '
                    f'F1={metrics["f1"]:.5f} TP={metrics["TP"]} '
                    f'FP={metrics["FP"]} FN={metrics["FN"]}\n')
    print('NER entity-level CoNLL-style metrics:',cli_scores)
    return errors


def run():
    if not (DATA/'pos_train.conll').exists():
        from .annotation import run as annotate
        annotate()
    vocab=json.loads((EMB/'word2idx.json').read_text('utf-8'))
    weights=np.load(EMB/'embeddings_w2v.npy')
    if len(vocab)!=weights.shape[0]:raise ValueError('C3 embedding/vocabulary mismatch')
    summary={}
    for task in ['pos','ner']:
        modes={};mode_tests={}
        for mode in ['frozen','finetuned']:
            frozen=(mode=='frozen')
            model,loaders,test,result=train_one(task,f'bilstm_{task}_{mode}',vocab,weights,
                                                 freeze=frozen,use_crf=(task=='ner'))
            modes[mode]=result;mode_tests[mode]=test
        bestmode=max(('finetuned','frozen'),key=lambda k:modes[k]['best_validation_f1'])
        best_test=mode_tests[bestmode]
        # Copy EXACTLY the checkpoint selected by validation F1; this avoids an
        # unnecessary, potentially non-identical third training run.
        import shutil
        shutil.copyfile(MODELS/f'bilstm_{task}_{bestmode}.pt',MODELS/f'bilstm_{task}.pt')
        best_history=modes[bestmode]['history']
        plot_curves(best_history,['train_loss','val_loss'],f'bilstm_{task}_loss.png',
                    f'bilstm_{task}: selected {bestmode} learning curves')
        if task=='pos':evaluate_pos(best_test)
        else:evaluate_ner(best_test)
        summary[task]={'modes':{k:{'val_f1':v['best_validation_f1'],'test_f1':v['test_f1']} for k,v in modes.items()},
                       'chosen_mode':bestmode,'test_metrics':best_test['metrics'],
                       'best_epoch':modes[bestmode]['best_epoch']}
    # Each ablation trained independently with identical data split, random seed and hyperparams.
    opts={
       'A1_unidirectional':dict(bidir=False,use_crf=True),
       'A2_no_dropout':dict(dropout=0.,use_crf=True),
       'A3_random_embeddings':dict(random_init=True,use_crf=True),
       'A4_softmax_no_crf':dict(use_crf=False),
    }
    ablation={}
    for name,cfg in opts.items():
        _,_,res,info=train_one('ner',name,vocab,weights,freeze=False,**cfg)
        ablation[name]={'validation_F1':info['best_validation_f1'],'test_F1':res['primary_f1'],
                        'description':{'A1_unidirectional':'No backward context',
                         'A2_no_dropout':'Dropout removed',
                         'A3_random_embeddings':'Replace C3 embeddings with random initialization',
                         'A4_softmax_no_crf':'Token softmax replaces CRF/Viterbi'}[name]}
    summary['ablations']=ablation
    summary['ner_crf_vs_softmax']={'crf':summary['ner']['test_metrics']['overall']['f1'],
                                   'softmax':ablation['A4_softmax_no_crf']['test_F1']}
    summary['ner_initialization_comparison']={
        'C3_word2vec_main_F1':summary['ner']['test_metrics']['overall']['f1'],
        'random_embeddings_A3_F1':ablation['A3_random_embeddings']['test_F1'],
        'label_reference':'Automated silver gazetteer labels, NOT human-reviewed gold',
        'changes':'Pretrained C3 global rescaling, minimum NER warm-up, TRAIN-only entity-weighted CE'
    }
    save_json('part2_summary.json',summary)
    return summary

if __name__=='__main__':run()


In [ ]:
%%writefile src/transformer.py
"""Part 3: handwritten four-layer pre-LN Transformer encoder + fair BiLSTM classifier."""
from __future__ import annotations
import math, copy, json, time
from collections import Counter
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from .common import (DATA, EMB, MODELS, FIGS, RES, TOPICS,load_articles,stratified_split,
                     seed_all,device,plot_curves,plot_confusion,save_json,show_counts)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

MAX_TOKENS=256; D_MODEL=128; HEADS=4; D_K=32; D_FF=512; NUM_LAYERS=4


class ScaledDotProductAttention(nn.Module):
    def __init__(self):super().__init__()
    def forward(self,q,k,v,key_mask=None):
        logits=(q@k.transpose(-2,-1))/math.sqrt(q.shape[-1])
        if key_mask is not None:
            # key_mask True=real token; broadcast over all query positions.
            logits=logits.masked_fill(~key_mask[:,None,:],-1e9)
        weights=F.softmax(logits,dim=-1)
        out=weights@v
        return out,weights


class MultiHeadSelfAttention(nn.Module):
    def __init__(self,d_model=D_MODEL,h=HEADS,dk=D_K):
        super().__init__();assert h*dk==d_model
        # Distinct nn.Linear per head, not one combined QKV matrix.
        self.q=nn.ModuleList([nn.Linear(d_model,dk) for _ in range(h)])
        self.k=nn.ModuleList([nn.Linear(d_model,dk) for _ in range(h)])
        self.v=nn.ModuleList([nn.Linear(d_model,dk) for _ in range(h)])
        self.attention=ScaledDotProductAttention();self.out=nn.Linear(h*dk,d_model)
    def forward(self,x,key_mask=None):
        features=[];attentions=[]
        for wq,wk,wv in zip(self.q,self.k,self.v):
            feature,att=self.attention(wq(x),wk(x),wv(x),key_mask)
            features.append(feature);attentions.append(att)
        return self.out(torch.cat(features,dim=-1)),torch.stack(attentions,dim=1)


class FeedForward(nn.Module):
    def __init__(self,d_model=D_MODEL,d_ff=D_FF,dropout=.1):
        super().__init__();self.w1=nn.Linear(d_model,d_ff);self.w2=nn.Linear(d_ff,d_model)
        self.drop=nn.Dropout(dropout)
    def forward(self,x):return self.w2(self.drop(F.relu(self.w1(x))))


class SinusoidalPositionEncoding(nn.Module):
    def __init__(self,d_model=D_MODEL,max_positions=MAX_TOKENS+1):
        super().__init__()
        p=torch.arange(max_positions,dtype=torch.float32).unsqueeze(1)
        div=torch.exp(torch.arange(0,d_model,2,dtype=torch.float32)*(-math.log(10000.)/d_model))
        encoding=torch.zeros(1,max_positions,d_model,dtype=torch.float32)
        encoding[0,:,0::2]=torch.sin(p*div)
        encoding[0,:,1::2]=torch.cos(p*div)
        self.register_buffer('pe',encoding,persistent=True)  # fixed, non-learned
    def forward(self,x):return x+self.pe[:,:x.size(1)]


class PreLNEncoderBlock(nn.Module):
    def __init__(self,d_model=D_MODEL,d_ff=D_FF,dropout=.1):
        super().__init__();self.ln1=nn.LayerNorm(d_model);self.ln2=nn.LayerNorm(d_model)
        self.mhsa=MultiHeadSelfAttention(d_model);self.ffn=FeedForward(d_model,d_ff,dropout)
        self.drop=nn.Dropout(dropout)
    def forward(self,x,mask=None):
        out,att=self.mhsa(self.ln1(x),mask)
        x=x+self.drop(out)
        x=x+self.drop(self.ffn(self.ln2(x)))
        return x,att


class TopicTransformer(nn.Module):
    def __init__(self,vocab_size,num_classes=5):
        super().__init__();self.embed=nn.Embedding(vocab_size,D_MODEL,padding_idx=0)
        self.cls_token=nn.Parameter(torch.empty(1,1,D_MODEL));nn.init.normal_(self.cls_token,std=.02)
        self.pe=SinusoidalPositionEncoding(D_MODEL)
        self.blocks=nn.ModuleList([PreLNEncoderBlock() for _ in range(NUM_LAYERS)])
        self.norm=nn.LayerNorm(D_MODEL)
        self.classifier=nn.Sequential(nn.Linear(D_MODEL,64),nn.ReLU(),nn.Linear(64,num_classes))
    def forward(self,tokens,mask,return_attention=False):
        B=tokens.shape[0]
        x=self.embed(tokens)*math.sqrt(D_MODEL)
        x=torch.cat([self.cls_token.expand(B,-1,-1),x],dim=1)
        m=torch.cat([torch.ones(B,1,dtype=torch.bool,device=mask.device),mask],dim=1)
        x=self.pe(x)
        attention=None
        for block in self.blocks:x,attention=block(x,m)
        logits=self.classifier(self.norm(x[:,0]))
        return (logits,attention) if return_attention else logits


class BiLSTMTopicClassifier(nn.Module):
    """A separate article classifier: sequence tagger F1 is not comparable to article accuracy."""
    def __init__(self,weights):
        super().__init__();self.embed=nn.Embedding.from_pretrained(torch.tensor(weights,dtype=torch.float32),
                                                                     freeze=False,padding_idx=0)
        self.lstm=nn.LSTM(weights.shape[1],64,batch_first=True,bidirectional=True,num_layers=2,dropout=.5)
        self.drop=nn.Dropout(.5);self.fc=nn.Sequential(nn.Linear(128,64),nn.ReLU(),nn.Linear(64,5))
    def forward(self,x,mask):
        length=mask.sum(1).clamp(min=1).cpu()
        packed=nn.utils.rnn.pack_padded_sequence(self.embed(x),length,batch_first=True,enforce_sorted=False)
        _,(h,_)=self.lstm(packed)
        out=torch.cat([h[-2],h[-1]],dim=-1)
        return self.fc(self.drop(out))


class ArticleDataset(Dataset):
    def __init__(self,articles,vocab,indices):
        self.items=[articles[i] for i in indices];self.vocab=vocab
    def __len__(self):return len(self.items)
    def __getitem__(self,i):
        s=self.items[i]
        words=s['tokens'][:MAX_TOKENS]
        ids=[self.vocab.get(w,1) for w in words]
        x=np.zeros(MAX_TOKENS,dtype=np.int64);x[:len(ids)]=ids
        return torch.tensor(x),torch.tensor(TOPICS.index(s['topic']),dtype=torch.long),words,s['id']


def batch_collate(batch):
    xs,ys,words,ident=zip(*batch)
    x=torch.stack(xs);y=torch.stack(ys)
    return x,y,x.ne(0),list(words),list(ident)


def data_loaders():
    articles=load_articles();vocab=json.loads((EMB/'word2idx.json').read_text('utf-8'))
    categories=[a['topic'] for a in articles]
    if set(categories)!=set(TOPICS):
        raise ValueError('All 5 topic classes must be present; found '+str(Counter(categories)))
    tr,va,te=stratified_split(articles,categories)
    split={s:show_counts(s,categories,inds) for s,inds in [('train',tr),('val',va),('test',te)]}
    save_json('transformer_category_distribution.json',split)
    loaders={name:DataLoader(ArticleDataset(articles,vocab,ids),batch_size=12,
                             shuffle=name=='train',collate_fn=batch_collate)
             for name,ids in [('train',tr),('val',va),('test',te)]}
    return articles,vocab,loaders


def predict(model,loader,dev,return_rows=False):
    model.eval();all_y=[];all_p=[];all_rows=[];loss_sum=0.;n=0
    with torch.no_grad():
        for x,y,m,tokens,idx in loader:
            x=x.to(dev);y=y.to(dev);m=m.to(dev)
            z=model(x,m);loss_sum+=float(F.cross_entropy(z,y).item());n+=1
            pred=z.argmax(1).cpu().tolist();all_p.extend(pred);all_y.extend(y.cpu().tolist())
            if return_rows:
                for i in range(len(idx)):
                    all_rows.append({'article_id':idx[i],'tokens':tokens[i],
                                     'gold':int(y[i]),'pred':int(pred[i])})
    met={'accuracy':float(accuracy_score(all_y,all_p)),
         'macro_f1':float(f1_score(all_y,all_p,average='macro',labels=list(range(5)),zero_division=0)),
         'loss':loss_sum/max(n,1)}
    return met,all_y,all_p,all_rows


def cosine_warmup_scheduler(optimizer,total_steps,warmup_steps=50):
    def fac(step):
        step+=1
        if step<=warmup_steps:return step/max(1,warmup_steps)
        progress=min(1.,(step-warmup_steps)/max(1,total_steps-warmup_steps))
        return .5*(1.+math.cos(math.pi*progress))
    return torch.optim.lr_scheduler.LambdaLR(optimizer,lr_lambda=fac)


def train_classifier(model,loaders,name,transformer=False):
    seed_all(42);dev=device();model=model.to(dev)
    lr=.0005;optimizer=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=.01)
    epochs=20;total_steps=epochs*len(loaders['train']);scheduler=cosine_warmup_scheduler(optimizer,total_steps,50)
    history=[];best=-1.;beststate=None;bestep=None
    for ep in range(1,epochs+1):
        model.train();t0=time.perf_counter();tl=0.;total=0;correct=0
        for x,y,m,_,_ in loaders['train']:
            x=x.to(dev);y=y.to(dev);m=m.to(dev)
            optimizer.zero_grad(set_to_none=True);z=model(x,m)
            loss=F.cross_entropy(z,y);loss.backward();nn.utils.clip_grad_norm_(model.parameters(),1.)
            optimizer.step();scheduler.step()
            tl+=float(loss.item())*len(y);total+=len(y);correct+=int((z.argmax(1)==y).sum())
        if dev.type=='cuda':torch.cuda.synchronize()
        secs=time.perf_counter()-t0
        vm,_,_,_=predict(model,loaders['val'],dev)
        history.append({'epoch':ep,'train_loss':tl/total,'val_loss':vm['loss'],
                        'train_accuracy':correct/total,'val_accuracy':vm['accuracy'],
                        'val_macro_f1':vm['macro_f1'],'seconds_per_epoch':round(secs,3)})
        print(name,'epoch',ep,'train acc',round(correct/total,3),'val acc',round(vm['accuracy'],3))
        if vm['macro_f1']>best+1e-8:
            best=vm['macro_f1'];beststate=copy.deepcopy(model.state_dict());bestep=ep
    # Requirement: all 20 epochs trained. Report checkpoint selected by validation only.
    model.load_state_dict(beststate)
    met,ys,ps,rows=predict(model,loaders['test'],dev,return_rows=True)
    torch.save({'model_state_dict':model.state_dict(),'best_epoch':bestep,
                'architecture':name,'num_classes':5},MODELS/f'{name}.pt')
    plot_curves(history,['train_loss','val_loss'],f'{name}_loss.png',name+': loss across 20 epochs')
    plot_curves(history,['train_accuracy','val_accuracy'],f'{name}_accuracy.png',name+': accuracy across 20 epochs')
    cm=confusion_matrix(ys,ps,labels=list(range(5)))
    plot_confusion(cm,TOPICS,f'{name}_confusion.png',name+': test confusion matrix')
    out={'test':met,'best_epoch_by_validation_F1':bestep,'best_val_F1':best,
         'training_seconds_per_epoch':round(float(np.mean([r['seconds_per_epoch'] for r in history])),3),
         'history':history,'confusion_matrix':cm.tolist()}
    save_json(f'{name}_metrics.json',out)
    return model,out,rows


def attention_figures(model,rows,vocab):
    dev=device();model.eval()
    correct=[r for r in rows if r['pred']==r['gold']]
    if len(correct)<3:
        print('WARNING: fewer than 3 correct test articles. Do not fabricate attention heatmaps.')
    figures=[];observations=[]
    for pos,r in enumerate(correct[:3],1):
        words=r['tokens'][:31]
        ids=[vocab.get(w,1) for w in words]+[0]*(MAX_TOKENS-len(words))
        x=torch.tensor(ids,device=dev,dtype=torch.long)[None,:];mask=x.ne(0)
        with torch.no_grad():
            z,att=model(x,mask,return_attention=True)
        # Two heads of last encoder layer; show first 32 tokens including CLS.
        n=len(words)+1
        weights=att.detach().cpu().numpy()[0,:,:n,:n]
        names=['[CLS]']+words
        for head in [0,1]:
            # CLS is the classification aggregator; inspect its top attended non-CLS keys.
            strongest=np.argsort(weights[head,0,1:])[::-1][:3]+1
            observations.append({'article_id':r['article_id'],'head':head+1,
                                 'top_cls_attention_words':[names[int(j)] for j in strongest],
                                 'attention_probabilities':[float(weights[head,0,int(j)]) for j in strongest]})
            fig,ax=plt.subplots(figsize=(max(8,n*.35),max(7,n*.32)))
            im=ax.imshow(weights[head],cmap='viridis',aspect='auto',vmin=0)
            fig.colorbar(im,ax=ax,label='Attention probability')
            ax.set_xticks(range(n),names,rotation=90,fontsize=7)
            ax.set_yticks(range(n),names,fontsize=7)
            ax.set(xlabel='Key token',ylabel='Query token',
                   title=f'Test article {r["article_id"]}: final-layer attention head {head+1} (correct: {TOPICS[r["gold"]]})')
            fig.tight_layout();filename=f'attention_article{pos}_head{head+1}.png';fig.savefig(FIGS/filename,dpi=180);plt.close(fig)
            figures.append(filename)
    return figures,observations


def comparison_paragraphs(bi,tr,observations):
    a=bi['test']['accuracy'];b=tr['test']['accuracy'];winner='Transformer' if b>a else ('BiLSTM' if a>b else 'Neither model')
    diff=abs(b-a)*100
    faster='BiLSTM' if bi['training_seconds_per_epoch']<tr['training_seconds_per_epoch'] else 'Transformer'
    converge='BiLSTM' if bi['best_epoch_by_validation_F1']<tr['best_epoch_by_validation_F1'] else ('Transformer' if bi['best_epoch_by_validation_F1']>tr['best_epoch_by_validation_F1'] else 'Neither')
    # These descriptions are computed from the *actual* final-layer attention weights.
    if observations:
        focus0=', '.join(observations[0]['top_cls_attention_words'])
        focus1=', '.join(observations[1]['top_cls_attention_words']) if len(observations)>1 else 'not available'
        att_sentence=f'For the first correctly classified article, the final layer CLS query had high attention on {focus0} in head 1.'
        att_second=f'The corresponding second head highlighted {focus1}; inspecting their differences can reveal whether heads distribute attention differently.'
    else:
        att_sentence='No correct held-out predictions were available for qualitative attention interpretation.'
        att_second='Therefore no attention-specific word-importance claim is justified for this run.'
    # 12 sentences, all requested points, honest about what attention can and cannot prove.
    sentences=[
      f'Both models were evaluated on the same stratified 5-class BBC Urdu article split.',
      f'The BiLSTM achieved {a*100:.2f}% test accuracy, while the Transformer achieved {b*100:.2f}%.',
      f'{winner} had the higher accuracy, with an absolute difference of {diff:.2f} percentage points.',
      f'The BiLSTM reached its best validation macro-F1 at epoch {bi["best_epoch_by_validation_F1"]}.',
      f'The Transformer reached its best validation macro-F1 at epoch {tr["best_epoch_by_validation_F1"]}.',
      f'{converge} reached its selected validation checkpoint in fewer epochs.',
      f'The average BiLSTM epoch took {bi["training_seconds_per_epoch"]:.2f} seconds, compared with {tr["training_seconds_per_epoch"]:.2f} seconds for the Transformer.',
      f'{faster} was faster per epoch in the measured runtime, and the difference can reflect recurrent sequential processing versus attention-matrix computation and the different parameter counts.',
      att_sentence,
      att_second,
      'For a dataset of just 200 to 300 articles, a smaller BiLSTM may be the safer first model because it has a lower capacity and can overfit less easily.',
      'However, the preferred architecture should be selected from repeated stratified evaluations, not assumed based on architecture alone.'
    ]
    assert len(sentences)==12
    (RES/'bilstm_vs_transformer_12_sentences.txt').write_text(' '.join(sentences),'utf-8')
    return sentences


def run():
    seed_all(42)
    articles,vocab,loaders=data_loaders()
    if not (EMB/'embeddings_w2v.npy').exists():raise FileNotFoundError('Run Part 1 first (C3 embeddings)')
    bi,bi_results,bi_rows=train_classifier(BiLSTMTopicClassifier(np.load(EMB/'embeddings_w2v.npy')),
                                         loaders,'bilstm_topic')
    tr,tr_results,tr_rows=train_classifier(TopicTransformer(len(vocab)),loaders,'transformer_cls',transformer=True)
    figures,attention_observations=attention_figures(tr,tr_rows,vocab)
    comparison=comparison_paragraphs(bi_results,tr_results,attention_observations)
    summary={'bilstm':bi_results['test'],'transformer':tr_results['test'],
             'bilstm_selected_epoch':bi_results['best_epoch_by_validation_F1'],
             'transformer_selected_epoch':tr_results['best_epoch_by_validation_F1'],
             'attention_heatmaps':figures,'attention_observations':attention_observations,
             'comparison_sentences':comparison}
    save_json('part3_summary.json',summary)
    return summary

if __name__=='__main__':run()

In [ ]:
%%writefile src/make_report.py
"""Create a 3-page evidence-based report.pdf; unfinished stages are explicitly marked."""
from __future__ import annotations
import json, os
from pathlib import Path
from reportlab.lib import colors
from reportlab.lib.enums import TA_CENTER
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib.pagesizes import A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from .common import ROOT, RES, TOPICS


def read(name):
    path=RES/name
    return json.loads(path.read_text('utf-8')) if path.exists() else None


def font_setup():
    path=os.environ.get('TIMES_NEW_ROMAN_TTF','')
    if path and Path(path).exists():
        pdfmetrics.registerFont(TTFont('SubmissionTimes',path))
        return 'SubmissionTimes'
    return 'Times-Roman'


def fmt(v,percent=False):
    if v is None:return 'Not evaluated'
    try:return f'{float(v)*100:.2f}%' if percent else f'{float(v):.4f}'
    except (ValueError,TypeError):return str(v)


def run():
    available={
        'part1':read('four_condition_comparison.json'),
        'part2':read('part2_summary.json'),
        'part3':read('part3_summary.json'),
    }
    unfinished=[name for name,content in available.items() if content is None]
    font=font_setup()
    body=ParagraphStyle('Body',fontName=font,fontSize=12,leading=18,spaceAfter=7)
    head=ParagraphStyle('Head',fontName=font,fontSize=12,leading=18,spaceBefore=8,spaceAfter=6)
    title=ParagraphStyle('Title',fontName=font,fontSize=14,leading=20,alignment=TA_CENTER,spaceAfter=8)
    note=ParagraphStyle('Note',fontName=font,fontSize=10,leading=14,spaceAfter=5,textColor=colors.HexColor('#7f2525'))
    small=ParagraphStyle('Small',fontName=font,fontSize=10,leading=14,spaceAfter=5)
    story=[]
    def p(x,style=body):story.append(Paragraph(str(x).replace('&','&amp;'),style))
    def h(x):p('<b>'+x+'</b>',head)
    def tbl(rows,widths=None):
        t=Table([[Paragraph(str(c).replace('&','&amp;'),small) for c in row] for row in rows],colWidths=widths,hAlign='LEFT',repeatRows=1)
        t.setStyle(TableStyle([('BACKGROUND',(0,0),(-1,0),colors.HexColor('#e7edf2')),
                               ('GRID',(0,0),(-1,-1),.3,colors.HexColor('#929aa0')),
                               ('VALIGN',(0,0),(-1,-1),'TOP'),('BOTTOMPADDING',(0,0),(-1,-1),3),
                               ('TOPPADDING',(0,0),(-1,-1),3)]))
        story.append(t);story.append(Spacer(1,6))
    p('DS 5007: NLP Assignment 2 — BBC Urdu Neural Pipeline',title)
    p('Student ID: 25i8030  |  Section: verify before submission',small)
    if unfinished:p('<b>INCOMPLETE DRAFT: real-data experiments missing ('+', '.join(unfinished)+'). Do not submit as a results report.</b>',note)
    h('Overview')
    p('This continuation of the BBC Urdu project examines whether word representations and neural sequence models preserve useful linguistic and topic information. The implementation uses the Assignment 1 corpus, a transparent rule-based annotation scheme, and PyTorch modules built without pretrained Transformer components. All reported test values below must originate from saved outputs, never illustrative numbers.')
    h('Part 1 Results — Embeddings (25 marks)')
    tf=read('tfidf_top_words.json');anal=read('analogies.json')
    if tf:
        for topic in TOPICS:
            words=', '.join(r['word'] for r in tf.get(topic,[])[:5])
            p(f'<b>{topic}:</b> {words or "No labelled articles"}.',small)
    else:p('TF-IDF topic word results: PENDING corpus-derived TF-IDF output.',small)
    if available['part1']:
        rows=[['Condition','MRR (eligible pairs)','Pairs scored']]
        for key,x in available['part1'].items():
            rows.append([key,fmt(x['MRR']),str(x['evaluable_pairs'])+'/20'])
        tbl(rows,[125,155,115])
        if anal:
            valid=[a for a in anal if a['correct_top3'] is not None]
            ncorrect=sum(a['correct_top3'] is True for a in valid)
            p(f'Analogy top-3 correctness: {ncorrect}/{len(valid)} evaluable tests; out-of-vocabulary cases are separately flagged.')
        p('The symmetric five-token-window PPMI vectors and Skip-gram embeddings were compared using the same cosine-ranking protocol. The C2 model uses raw text; C3 uses cleaned text; C4 doubles dimension from 100 to 200. Examine both pair coverage and MRR before identifying the strongest setting.')
    else:p('C1–C4 nearest neighbours, 20-pair MRR, training curves and 10 analogy outcomes: PENDING original raw.txt and cleaned.txt.')
    p('Interpretation caveat: a larger embedding dimension adds capacity, but it does not guarantee better semantic neighbours. TF-IDF is an article-level representation, while PPMI and Word2Vec estimate local lexical associations.')

    story.append(PageBreak())
    h('Part 2 Results — POS Tagging and Named Entities (25 marks)')
    ann=read('annotation_summary.json')
    if ann:
        p('Five hundred sampled sentences were stratified by topic. Split sizes: '+', '.join(f'{k}={v}' for k,v in ann['split_sizes'].items())+'.')
    else:p('Sentence annotation, label distributions and 70/15/15 split: PENDING; original corpus and metadata not provided.')
    p('POS/NER annotations start as rule-generated silver labels, not independently verified human gold. The assignment lists eleven POS names despite saying twelve; PROPN is recorded as an explicit twelfth class. Human review is necessary before treating held-out scores as a reliable estimate of tagging quality.')
    s2=available['part2']
    if s2:
        rows=[['Task / mode','Validation F1','Test F1']]
        for task in ('pos','ner'):
            for mode,stats in s2[task]['modes'].items():
                rows.append([task.upper()+' / '+mode,fmt(stats['val_f1']),fmt(stats['test_f1'])])
        tbl(rows,[135,125,125])
        pos=s2['pos']['test_metrics'];ner=s2['ner']['test_metrics']
        p(f'POS test token accuracy = {fmt(pos["accuracy"],True)}; macro-F1 = {fmt(pos["macro_f1"])}. NER strict entity-level overall P/R/F1 = {fmt(ner["overall"]["precision"])}, {fmt(ner["overall"]["recall"])}, {fmt(ner["overall"]["f1"])}.')
        p(f'NER F1 with CRF = {fmt(s2["ner_crf_vs_softmax"]["crf"])} and with softmax = {fmt(s2["ner_crf_vs_softmax"]["softmax"])}.')
        rows=[['Ablation','Test NER entity F1']]
        for name,r in s2['ablations'].items():rows.append([name,fmt(r['test_F1'])])
        tbl(rows,[235,150])
        p('A1 tests backward context; A2 tests dropout; A3 replaces the C3 initialization; and A4 removes structured CRF decoding. A difference in measured F1 does not itself establish a mechanism without examining example errors and variability.')
    else:
        p('Frozen/fine-tuned BiLSTM performance, CRF/Viterbi comparison, all four ablations and confusion/error examples: PENDING actual training.')
    p('For the three most frequently confused POS tag pairs, inspect two original sentences each. For NER, manually review at least five false positives and five false negatives, checking both boundaries and entity types.')

    story.append(PageBreak())
    h('Part 3 Results — Transformer Topic Classifier (20 marks)')
    s3=available['part3']
    if s3:
        b=s3['bilstm'];t=s3['transformer']
        tbl([['Model','Test accuracy','Test macro-F1'],
             ['BiLSTM topic classifier',fmt(b['accuracy'],True),fmt(b['macro_f1'])],
             ['4-block Transformer',fmt(t['accuracy'],True),fmt(t['macro_f1'])]],
            [170,115,110])
        p(f"Final encoder-layer attention heatmaps: {len(s3['attention_heatmaps'])} figures generated from correctly classified test articles, using two attention heads per article.")
        p('BiLSTM vs Transformer comparison (12 sentences):',small)
        for sentence in s3['comparison_sentences']:
            p(sentence,small)
    else:
        p('Five-class distribution, 20-epoch accuracy/loss curves, test confusion matrix, and six final-layer attention heatmaps: PENDING actual training.')
        p('The classifier uses independently projected four-head attention, a fixed sinusoidal positional buffer, four pre-normalized residual encoder blocks, a learned [CLS] token and a two-layer MLP classification head. A separate BiLSTM article classifier is required for a like-for-like task comparison.')
    h('Conclusion')
    if not unfinished:
        p('The results above provide an auditable comparison of sparse and dense word representations, structured and unstructured sequence decoding, and recurrent versus attention-based topic classification. Error analysis and manual review remain necessary to understand the limitations of automatically created annotations.')
    else:
        p('No test accuracy or comparison can be concluded until cleaned.txt, raw.txt and Metadata.json are supplied, models are trained, and notebook outputs are verified. This draft intentionally does not invent experimental results.')
    p('GitHub repository URL: ADD PUBLIC REPOSITORY URL AFTER PUSHING THE REAL PROJECT',small)
    target=ROOT/'report.pdf'
    doc=SimpleDocTemplate(str(target),pagesize=A4,leftMargin=44,rightMargin=44,topMargin=43,
                          bottomMargin=40,title='DS5007 Assignment 2 report'+(' — INCOMPLETE' if unfinished else ''))
    def decorate(canvas,doc):
        canvas.saveState();canvas.setFont(font,9)
        canvas.drawRightString(A4[0]-44,23,f'DS5007 • Page {doc.page}');canvas.restoreState()
    doc.build(story,onFirstPage=decorate,onLaterPages=decorate)
    print('Report:',target,'incomplete stages:',unfinished)
    return target

if __name__=='__main__':run()

In [ ]:
%%writefile assets/analogies.json
[
 {"a":"پاکستان","b":"اسلام آباد","c":"بھارت","gold":"دہلی"},
 {"a":"پاکستان","b":"اسلام آباد","c":"افغانستان","gold":"کابل"},
 {"a":"پاکستان","b":"اسلام آباد","c":"چین","gold":"بیجنگ"},
 {"a":"پاکستان","b":"کراچی","c":"بھارت","gold":"ممبئی"},
 {"a":"پاکستان","b":"روپیہ","c":"امریکہ","gold":"ڈالر"},
 {"a":"پاکستان","b":"اردو","c":"ایران","gold":"فارسی"},
 {"a":"پاکستان","b":"پاکستانی","c":"بھارت","gold":"بھارتی"},
 {"a":"سکول","b":"استاد","c":"ہسپتال","gold":"ڈاکٹر"},
 {"a":"عدالت","b":"جج","c":"سکول","gold":"استاد"},
 {"a":"فٹبال","b":"گول","c":"کرکٹ","gold":"وکٹ"}
]

In [ ]:
%%writefile assets/mrr_20_pairs.json
[
 {"query":"پاکستان","gold":"اسلام آباد"},
 {"query":"بھارت","gold":"دہلی"},
 {"query":"افغانستان","gold":"کابل"},
 {"query":"چین","gold":"بیجنگ"},
 {"query":"پاکستان","gold":"کراچی"},
 {"query":"بھارت","gold":"ممبئی"},
 {"query":"سکول","gold":"استاد"},
 {"query":"ہسپتال","gold":"ڈاکٹر"},
 {"query":"عدالت","gold":"جج"},
 {"query":"کرکٹ","gold":"میچ"},
 {"query":"فٹبال","gold":"کھلاڑی"},
 {"query":"حکومت","gold":"وزیر"},
 {"query":"معیشت","gold":"بجٹ"},
 {"query":"کاروبار","gold":"تجارت"},
 {"query":"تعلیم","gold":"طالب علم"},
 {"query":"صحت","gold":"ہسپتال"},
 {"query":"انتخاب","gold":"ووٹ"},
 {"query":"بینک","gold":"روپے"},
 {"query":"فوج","gold":"سپاہی"},
 {"query":"بارش","gold":"سیلاب"}
]

In [ ]:
%%writefile tests/test_components.py
"""Mathematical smoke tests only: never mistaken for BBC Urdu experiment results."""
import unittest, tempfile, itertools
from pathlib import Path
from unittest.mock import patch
import numpy as np
import torch
from scipy import sparse
from src.common import stratified_split
from src.annotation import build_lexicon,gazetteer,ner_bio,POS_TAGS
from src.embeddings import ppmi_matrix,cosine_neighbors,SkipGram,tfidf
from src.sequence import LinearChainCRF, get_spans, BiLSTMTagger, collate,conlleval_scores
from src.transformer import ScaledDotProductAttention,TopicTransformer,SinusoidalPositionEncoding, MultiHeadSelfAttention

class TestComponents(unittest.TestCase):
 def test_lexicon_gazetteer(self):
    _,counts=build_lexicon()
    self.assertTrue(all(counts[tag]>=200 for tag in ('NOUN','VERB','ADJ')))
    g=gazetteer();self.assertTrue(len(g)>=130);self.assertEqual(len(POS_TAGS),12)
    tokens='imran khan visited islamabad'.split();tags=ner_bio(tokens,g)
    self.assertEqual(tags[:2],['B-PER','I-PER']);self.assertEqual(tags[-1],'B-LOC')
 def test_tfidf_formula_and_ppmi(self):
    docs=[['a','b','a'],['b','c'],['a','c']]
    vocab={'<pad>':0,'<unk>':1,'a':2,'b':3,'c':4}
    with tempfile.TemporaryDirectory() as temp:
     root=Path(temp)
     with patch('src.embeddings.EMB',root),patch('src.embeddings.save_json',lambda *args:None):
      tfidf([{'tokens':d,'topic':'Politics'} for d in docs],vocab)
      dense=np.load(root/'tfidf_matrix.npy')
      self.assertAlmostEqual(dense[0,2],2*np.log(3/3),places=6)
      self.assertAlmostEqual(dense[0,3],np.log(3/3),places=6)
      ppmi=ppmi_matrix(docs,vocab)
      self.assertTrue(np.isfinite(ppmi.data).all())
      self.assertTrue((ppmi.data>=0).all())
      self.assertEqual(ppmi.shape,(5,5))
      self.assertTrue((root/'ppmi_matrix.npy').exists())
 def test_skipgram_bce(self):
    m=SkipGram(20,100)
    z=m(torch.tensor([3,4]),torch.tensor([6,7]),torch.randint(1,19,(2,10)))
    self.assertTrue(torch.isfinite(z));z.backward()
    self.assertIsNotNone(m.V.weight.grad)
 def test_crf_matches_brute(self):
    torch.manual_seed(3)
    crf=LinearChainCRF(3)
    em=torch.randn(1,4,3)
    mask=torch.ones(1,4,dtype=torch.bool)
    g=torch.tensor([[0,2,1,2]])
    logz=crf.log_partition(em,mask).item()
    scores=[]
    for tup in itertools.product(range(3),repeat=4):
      score=crf.start[tup[0]]+em[0,0,tup[0]]
      for t in range(1,4):score=score+crf.transitions[tup[t-1],tup[t]]+em[0,t,tup[t]]
      score=score+crf.end[tup[-1]]
      scores.append((score.item(),tup))
    expected=torch.logsumexp(torch.tensor([x[0] for x in scores]),dim=0).item()
    self.assertAlmostEqual(logz,expected,places=5)
    want=max(scores)[1];got=tuple(crf.viterbi(em,mask)[0]);self.assertEqual(got,want)
    crf(em,g,mask).backward();self.assertIsNotNone(crf.transitions.grad)
 def test_bio_span_matching(self):
    true=[['B-PER','I-PER','O','B-LOC']]
    pred=[['B-PER','I-PER','O','O']]
    scores,fp,fn=conlleval_scores(true,pred)
    self.assertEqual(scores['PER']['TP'],1);self.assertEqual(scores['LOC']['FN'],1)
    self.assertEqual(len(fn),1)
 def test_attn_mask_and_cls(self):
    att=ScaledDotProductAttention()
    x=torch.randn(2,6,32)
    out,w=att(x,x,x,torch.tensor([[1,1,1,0,0,0],[1,1,1,1,0,0]],dtype=torch.bool))
    self.assertEqual(out.shape,(2,6,32));self.assertLess(w[0,:,-1].max().item(),1e-8)
    self.assertTrue(torch.allclose(w.sum(-1),torch.ones(2,6),atol=1e-5))
    self.assertFalse(any('pe' in n for n,_ in SinusoidalPositionEncoding().named_parameters()))
    model=TopicTransformer(70)
    tok=torch.randint(1,69,(1,8));tok[0,-2:]=0
    res,weights=model(tok,tok.ne(0),return_attention=True)
    self.assertEqual(res.shape,(1,5));self.assertEqual(weights.shape,(1,4,9,9))
    self.assertEqual(len(model.blocks),4)
 def test_split(self):
    labels=['Politics']*80+['Sports']*80+['Economy']*80+['International']*80+['Health & Society']*80
    splits=stratified_split(list(range(400)),labels)
    self.assertEqual([len(s) for s in splits],[280,60,60])
    self.assertEqual(set(splits[0])&set(splits[2]),set())

if __name__=='__main__':unittest.main()

## Dataset check
Confirm that article IDs and file counts match before training.

In [ ]:
import json
from src.common import split_numbered_articles, load_metadata

raw_articles = split_numbered_articles((DATA_DIR / 'raw.txt').read_text(encoding='utf-8-sig'))
clean_articles = split_numbered_articles((DATA_DIR / 'cleaned.txt').read_text(encoding='utf-8-sig'))
metadata = load_metadata()

assert raw_articles is not None and clean_articles is not None, 'Expected numbered ### ARTICLE N ### format.'
raw_ids = {a for a, _ in raw_articles}
clean_ids = {a for a, _ in clean_articles}
meta_ids = {int(m['article_number']) for m in metadata}
assert len(raw_ids) == len(raw_articles), 'Duplicate article IDs in raw.txt'
assert len(clean_ids) == len(clean_articles), 'Duplicate article IDs in cleaned.txt'
assert len(meta_ids) == len(metadata), 'Duplicate metadata IDs'
assert raw_ids == clean_ids == meta_ids, 'Article numbering mismatch between dataset files'
assert len(raw_ids) >= 200, 'Assignment corpus should contain at least 200 real articles'

print('DATASET VALIDATION: PASS')
print('Article count:', len(raw_ids))
print('raw.txt articles:', len(raw_articles))
print('cleaned.txt articles:', len(clean_articles))
print('Metadata.json records:', len(metadata))
print('Article IDs aligned: YES')
print('Category review CSV:', 'PRESENT' if (DATA_DIR/'topic_labels_review.csv').exists() else 'NOT FOUND')
print('Next: Start Assignment 2 Part 1 below.')

---
## Assignment 2 — Experiments
| Part | Topic | Marks |
|:--|:--|--:|
| 1 | Word embeddings | 25 |
| 2 | POS tagging and NER | 25 |
| 3 | Transformer classification | 20 |
| 4 | GitHub submission | 5 |

## Part 1 — Word embeddings (25 marks)

TF-IDF, PPMI and three Skip-gram conditions. The code is unchanged from the previously executed notebook. Existing files are reused when available.


In [ ]:
# Part 1 — use saved genuine embeddings when present; otherwise train from scratch
from src import embeddings
from src.common import EMB, RES
needed=['embeddings_w2v.npy','word2idx.json','tfidf_matrix.npy','ppmi_matrix.npy']
if all((EMB/x).exists() for x in needed) and (RES/'four_condition_comparison.json').exists():
    print('Using previously trained Part 1 files in this runtime.')
else:
    print('Training Part 1 because saved embeddings are not present.')
    part1=embeddings.run()
print('Part 1 artifacts:',{x:(EMB/x).exists() for x in needed})


In [ ]:
# Part 1: numerical results and plots
from src.common import RES, FIGS
from IPython.display import Image, display
import json
for name in ['tfidf_top_words.json','ppmi_neighbors.json','w2v_neighbors.json',
             'analogies.json','four_condition_comparison.json']:
    f=RES/name
    print('\n##',name,'exists:',f.exists())
    if f.exists():print(json.dumps(json.loads(f.read_text('utf8')),ensure_ascii=False,indent=2)[:4000])
for name in ['ppmi_tsne.png','embeddings_w2v_loss.png']:
    f=FIGS/name
    if f.exists():display(Image(filename=str(f),width=650))
print('Part 1 inspection complete.')

## Part 2 — POS and Named Entity Recognition

The 500 sentences are drawn from the cleaned Urdu corpus. NER references come from a rule-based gazetteer, not human gold annotations. This run compares scaled C3 Word2Vec embeddings with random embeddings using consistent training settings.

The main NER model is chosen by validation F1, with fine-tuning preferred if validation scores tie. The same class weighting and warm-up are used in the A3 random-embedding comparison.


In [ ]:
# Check topic review before annotation and article classification.
import csv,collections
from src.common import DATA, TOPICS
with (DATA/'topic_labels_review.csv').open(encoding='utf-8-sig',newline='') as f:
    rows=list(csv.DictReader(f))
verified=collections.Counter(r.get('topic','').strip() for r in rows if r.get('topic','').strip() in TOPICS)
print('Manually reviewed topic counts:',dict(verified),'of',len(rows),'articles')
if sum(verified.values()) < 200 or len(verified)<5 or min(verified.values())<7:
    print('ACTION REQUIRED: Fill the topic column in data/topic_labels_review.csv with verified labels.')
    print('Five allowed categories:',TOPICS)
    print('You can run Part 1 without these labels; Part 2/3 need sufficient reviewed categories.')
else:
    print('Topic-count gate passed for Part 2/3. Continue after reviewing annotation quality.')

In [ ]:
# Part 2A: silver POS/NER CoNLL annotations; manually review before treating as references
from src import annotation
annotation_summary=annotation.run()
print('Lexicon sizes:',annotation_summary['lexicon_counts'])
print('Gazetteer seeds:',annotation_summary['gazetteer_seed_counts'])
print('Sentence split sizes:',annotation_summary['split_sizes'])
print('Topic distribution:',annotation_summary['topics'])

In [ ]:
# Check the C3 embedding scale and NER vocabulary coverage (training text only)
from pathlib import Path
from collections import Counter
import json, numpy as np
from src.common import EMB, DATA
vocab=json.loads((EMB/'word2idx.json').read_text(encoding='utf-8'))
w=np.load(EMB/'embeddings_w2v.npy')
train_words=[line.split('\t')[0] for line in (DATA/'ner_train.conll').read_text(encoding='utf-8').splitlines() if line.strip()]
unknown=sum(word.lower() not in vocab for word in train_words)
print('C3 embedding shape:',w.shape)
print('C3 original non-special-token std:',round(float(w[2:].std()),6))
print('PyTorch random embedding expected std: about 1.0')
print('NER training token OOV share:',f'{unknown}/{len(train_words)}',f'({100*unknown/max(1,len(train_words)):.1f}%)')


In [ ]:
# Part 2B — retrain corrected NER and compare Word2Vec vs A3 random embeddings
import importlib
from src import sequence
importlib.reload(sequence)
part2=sequence.run()
main=part2['ner']['test_metrics']['overall']
a3=part2['ablations']['A3_random_embeddings']['test_F1']
print('C3 Word2Vec NER:',{k:round(main[k],4) for k in ('precision','recall','f1')})
print('A3 Random NER F1:',round(a3,4))
print('Selected main model:',part2['ner']['chosen_mode'])
print('Scores use silver automatic labels; gold human NER evaluation not available.')


In [ ]:
# Diagnostics from the completed Part 2 run
import json
from src.common import RES
metrics=json.loads((RES/'part2_summary.json').read_text('utf-8'))
print('Main NER result:',metrics['ner']['test_metrics']['overall'])
print('A3 random embedding result:',metrics['ablations']['A3_random_embeddings'])
print('CRF vs token softmax:',metrics['ner_crf_vs_softmax'])
print('Selected NER mode:',metrics['ner']['chosen_mode'])


In [ ]:
# NER quick check — saved scores and prediction counts from the new run
from src.common import RES
import json
j=json.loads((RES/'part2_summary.json').read_text('utf-8'))
for name,entry in [('Main C3',j['ner']['test_metrics']['overall']),
                   ('A3 random',{'f1':j['ablations']['A3_random_embeddings']['test_F1']})]:
    print(name,':',entry)
print('Zero-F1 warning:',j['ner']['test_metrics']['overall']['f1']==0)
print('Scores are against automatically generated silver labels.')


In [ ]:
# Part 2: view output figures
for name in ['pos_confusion.png','bilstm_pos_loss.png','bilstm_ner_loss.png',
             'A1_unidirectional_loss.png','A2_no_dropout_loss.png',
             'A3_random_embeddings_loss.png','A4_softmax_no_crf_loss.png']:
    f=FIGS/name
    print(name, 'exists:',f.exists())
    if f.exists():display(Image(filename=str(f),width=650))
print('Part 2 result inspection complete.')

## Part 3 — Transformer Classification (20 marks)
Train a four-block Transformer for five Urdu news topics and compare it with a BiLSTM classifier. Report confusion matrices and attention maps.

In [ ]:
# Part 3: from-scratch Transformer and BiLSTM article topic baseline
from src import transformer
part3=transformer.run()
print('Transformer test:',part3['transformer'])
print('BiLSTM article classifier test:',part3['bilstm'])
print('Attention heatmaps:',part3['attention_heatmaps'])

In [ ]:
# Part 3: view plots and 12-sentence comparison
for name in ['bilstm_topic_loss.png','transformer_cls_loss.png',
             'bilstm_topic_accuracy.png','transformer_cls_accuracy.png',
             'bilstm_topic_confusion.png','transformer_cls_confusion.png']+part3['attention_heatmaps']:
    f=FIGS/name
    print(name,'exists:',f.exists())
    if f.exists():display(Image(filename=str(f),width=650))
f=RES/'bilstm_vs_transformer_12_sentences.txt'
print('Comparison:',f.read_text('utf8') if f.exists() else 'not yet generated')

## Report and model files

Generate the report **after** the revised Part 2 training finishes. Do not use the previous PDF for the new NER score.


In [ ]:
# Generate report once training has completed
from src import make_report
report_path=make_report.run()
print('Report:',report_path)
required=[ROOT/'embeddings'/x for x in ['tfidf_matrix.npy','ppmi_matrix.npy','embeddings_w2v.npy','word2idx.json']]
required +=[ROOT/'models'/x for x in ['bilstm_pos.pt','bilstm_ner.pt','transformer_cls.pt']]
required +=[ROOT/'data'/x for x in ['pos_train.conll','pos_test.conll','ner_train.conll','ner_test.conll']]
for f in required:print(('OK ' if f.exists() else 'MISSING ')+str(f.relative_to(ROOT)))
print('Fill GitHub URL in notebook/report before submission.')

## Export
Save the generated outputs. Keep the model checkpoints and embedding matrices alongside this notebook for submission.

In [ ]:
# Package completed output files; do not upload copyrighted full text to a public repo.
from pathlib import Path
import zipfile
bundle=ROOT/'DS5007_Assignment2_Outputs.zip'
with zipfile.ZipFile(bundle,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for folder in ['data','embeddings','models','figures','results']:
        for file in sorted((ROOT/folder).rglob('*')):
            if file.is_file():z.write(file,arcname=str(file.relative_to(ROOT)))
    if (ROOT/'report.pdf').exists():z.write(ROOT/'report.pdf',arcname='report.pdf')
print('Archive created:',bundle,'bytes:',bundle.stat().st_size)
print('Verify all cells have outputs and the actual training metrics exist before submission.')

In [ ]:
# Results summary (loaded from real experiment files; never hard-coded)
import json
from src.common import RES
p2file=RES/'part2_summary.json'
if not p2file.exists():
    print('Run the Part 2 training cell before requesting updated NER results.')
else:
    p2=json.loads(p2file.read_text(encoding='utf-8'))
    m=p2['ner']['test_metrics']['overall']
    a3=p2['ablations']['A3_random_embeddings']['test_F1']
    print('POS accuracy:',round(p2['pos']['test_metrics']['accuracy']*100,2),'%')
    print('NER (C3 Word2Vec + CRF) P/R/F1:',*[round(m[k],4) for k in ('precision','recall','f1')])
    print('NER (A3 random embeddings + CRF) F1:',round(a3,4))
    print('Main NER mode chosen by validation F1:',p2['ner']['chosen_mode'])
    print('Reference labels: automatically generated silver annotations, not independently reviewed gold.')
    p3=RES/'part3_summary.json'
    if p3.exists():
        raw=json.loads(p3.read_text(encoding='utf-8'))
        print('Part 3 results: see part3_summary.json')
